# Biomedical QA Dense Retrieval: Reproducible Knowledge Distillation Pipeline

This notebook provides a single end-to-end implementation for the experiments reported in the manuscript. It covers data preparation, leakage-controlled splitting, explicit negative sampling, No-KD and teacher baselines, K1/K2/K3 knowledge distillation, alpha sensitivity, repeated-seed experiments, retrieval evaluation, ranking-order analysis, statistical testing, loss/gradient auditing, and efficiency measurements.

All paths are intentionally generic. Set `PROJECT_DIR` and `MEDQUAD_CSV` in the configuration cell before execution. The notebook contains no user-specific paths or identifiers.

## 1. Environment and project configuration

The reference implementation targets Google Colab with an NVIDIA GPU. Public PubMedQA and BioASQ datasets are loaded from Hugging Face; MedQuAD is read from a user-supplied CSV file.

In [ ]:
!pip -q install -U datasets transformers accelerate sentencepiece einops scikit-learn

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/PROJECT_ROOT"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
SPLIT_DIR = os.path.join(PROJECT_DIR, "fixed_positive_splits")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
CONFIG_DIR = os.path.join(PROJECT_DIR, "configs")
MEDQUAD_CSV = os.path.join(DATA_DIR, "MedQuad.csv")

for directory in [PROJECT_DIR, DATA_DIR, SPLIT_DIR, RESULTS_DIR, CHECKPOINT_DIR, CONFIG_DIR]:
    os.makedirs(directory, exist_ok=True)

if not os.path.exists(MEDQUAD_CSV):
    raise FileNotFoundError(f"Set MEDQUAD_CSV to the local MedQuAD CSV path. Current value: {MEDQUAD_CSV}")

print("Project directory:", PROJECT_DIR)
print("MedQuAD file:", MEDQUAD_CSV)

In [ ]:
import re
import json
import math
import time
import gc
import random
import hashlib
import warnings
import unicodedata

from collections import defaultdict
from typing import Dict, List, Optional, Tuple, Union, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    get_linear_schedule_with_warmup
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

torch.cuda.empty_cache()

print("Seed:", SEED)
print("Device:", device)

## 2. Dataset loading and standardization

In [ ]:
pubmedqa = load_dataset(
    "qiaojin/PubMedQA",
    "pqa_labeled"
)

bioasq = load_dataset(
    "kroshan/BioASQ"
)

medquad_raw = pd.read_csv(
    MEDQUAD_CSV
)

print("PubMedQA:")
print(pubmedqa)

print("\nBioASQ:")
print(bioasq)

print("\nMedQuAD:")
print("Rows:", len(medquad_raw))
print("Columns:", medquad_raw.columns.tolist())

In [ ]:
# =========================================================
# BIOASQ ANSWER / CONTEXT STRUCTURE AUDIT
# =========================================================

import re
import pandas as pd


# ---------------------------------------------------------
# Collect raw BioASQ rows
# ---------------------------------------------------------

bioasq_raw_rows = []

for split_name in bioasq.keys():

    split_df = bioasq[
        split_name
    ].to_pandas()

    split_df["original_split"] = (
        split_name
    )

    bioasq_raw_rows.append(
        split_df
    )


bioasq_raw_df = pd.concat(
    bioasq_raw_rows,
    ignore_index=True
)


# ---------------------------------------------------------
# Parse <answer> and <context> fields
# ---------------------------------------------------------

def parse_bioasq_text(text):

    text = str(text)

    answer_match = re.search(
        r"<answer>\s*(.*?)\s*<context>",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    context_match = re.search(
        r"<context>\s*(.*)$",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    answer = (
        answer_match.group(1).strip()
        if answer_match
        else None
    )

    context = (
        context_match.group(1).strip()
        if context_match
        else None
    )

    return answer, context


parsed = (
    bioasq_raw_df["text"]
    .apply(parse_bioasq_text)
)

bioasq_raw_df[
    "parsed_answer"
] = parsed.apply(
    lambda x: x[0]
)

bioasq_raw_df[
    "parsed_context"
] = parsed.apply(
    lambda x: x[1]
)


# ---------------------------------------------------------
# Structural checks
# ---------------------------------------------------------

has_answer_tag = (
    bioasq_raw_df["text"]
    .astype(str)
    .str.contains(
        "<answer>",
        case=False,
        regex=False
    )
)

has_context_tag = (
    bioasq_raw_df["text"]
    .astype(str)
    .str.contains(
        "<context>",
        case=False,
        regex=False
    )
)

answer_tag_count = (
    bioasq_raw_df["text"]
    .astype(str)
    .str.count(
        "(?i)<answer>"
    )
)

context_tag_count = (
    bioasq_raw_df["text"]
    .astype(str)
    .str.count(
        "(?i)<context>"
    )
)


# ---------------------------------------------------------
# Report
# ---------------------------------------------------------

print("=" * 70)
print("BIOASQ ANSWER / CONTEXT STRUCTURE AUDIT")
print("=" * 70)

print(
    "Total raw rows:",
    len(bioasq_raw_df)
)

print(
    "Rows with <answer>:",
    int(has_answer_tag.sum())
)

print(
    "Rows with <context>:",
    int(has_context_tag.sum())
)

print(
    "Rows missing <answer>:",
    int((~has_answer_tag).sum())
)

print(
    "Rows missing <context>:",
    int((~has_context_tag).sum())
)

print(
    "Rows with multiple <answer> tags:",
    int((answer_tag_count > 1).sum())
)

print(
    "Rows with multiple <context> tags:",
    int((context_tag_count > 1).sum())
)

print(
    "Failed answer parses:",
    int(
        bioasq_raw_df[
            "parsed_answer"
        ].isna().sum()
    )
)

print(
    "Failed context parses:",
    int(
        bioasq_raw_df[
            "parsed_context"
        ].isna().sum()
    )
)

print(
    "Empty parsed answers:",
    int(
        bioasq_raw_df[
            "parsed_answer"
        ]
        .fillna("")
        .str.strip()
        .eq("")
        .sum()
    )
)

print(
    "Empty parsed contexts:",
    int(
        bioasq_raw_df[
            "parsed_context"
        ]
        .fillna("")
        .str.strip()
        .eq("")
        .sum()
    )
)


# ---------------------------------------------------------
# Show examples
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("PARSED EXAMPLES")
print("=" * 70)

sample_df = (
    bioasq_raw_df[
        [
            "question",
            "parsed_answer",
            "parsed_context"
        ]
    ]
    .drop_duplicates()
    .head(5)
)

for i, row in sample_df.iterrows():

    print("\nQUESTION:")
    print(
        str(row["question"])[:500]
    )

    print("\nANSWER:")
    print(
        str(row["parsed_answer"])[:500]
    )

    print("\nCONTEXT:")
    print(
        str(row["parsed_context"])[:1000]
    )

    print("-" * 70)

In [ ]:
# =========================================================
# PUBMEDQA CONTEXT LABEL AUDIT
# =========================================================

from collections import Counter
import pandas as pd


# ---------------------------------------------------------
# Inspect context segments and labels
# ---------------------------------------------------------

label_counter = Counter()

length_mismatches = []
empty_context_segments = []
empty_labels = []

examples = []

pubmedqa_train = pubmedqa["train"]

for idx, row in enumerate(pubmedqa_train):

    context_obj = row["context"]

    contexts = context_obj.get(
        "contexts",
        []
    )

    labels = context_obj.get(
        "labels",
        []
    )

    # Check whether each context segment has one label
    if len(contexts) != len(labels):
        length_mismatches.append({
            "row_index": idx,
            "pubid": row.get("pubid"),
            "num_contexts": len(contexts),
            "num_labels": len(labels)
        })

    # Count labels
    for label in labels:

        label_text = str(label).strip()

        if label_text == "":
            empty_labels.append(
                idx
            )
        else:
            label_counter[
                label_text
            ] += 1

    # Check empty context segments
    for segment_idx, segment in enumerate(
        contexts
    ):

        if str(segment).strip() == "":
            empty_context_segments.append({
                "row_index": idx,
                "segment_index": segment_idx
            })

    # Save a few examples
    if len(examples) < 5:

        examples.append({
            "question": row["question"],
            "contexts": contexts,
            "labels": labels
        })


# ---------------------------------------------------------
# Report
# ---------------------------------------------------------

print("=" * 70)
print("PUBMEDQA CONTEXT LABEL AUDIT")
print("=" * 70)

print(
    "Total PubMedQA rows:",
    len(pubmedqa_train)
)

print(
    "Rows with context/label length mismatch:",
    len(length_mismatches)
)

print(
    "Empty context segments:",
    len(empty_context_segments)
)

print(
    "Empty labels:",
    len(empty_labels)
)

print("\nLabel distribution:")

for label, count in sorted(
    label_counter.items(),
    key=lambda x: (-x[1], x[0])
):
    print(
        f"{label}: {count}"
    )


# ---------------------------------------------------------
# Show examples
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("CONTEXT / LABEL EXAMPLES")
print("=" * 70)

for example_index, example in enumerate(
    examples,
    start=1
):

    print(
        f"\nEXAMPLE {example_index}"
    )

    print("\nQUESTION:")
    print(
        str(example["question"])
    )

    print("\nSEGMENTS:")

    for segment_index, (
        segment,
        label
    ) in enumerate(
        zip(
            example["contexts"],
            example["labels"]
        ),
        start=1
    ):

        print(
            f"\nSegment {segment_index}"
        )

        print(
            f"Label: {label}"
        )

        print(
            str(segment)[:700]
        )

    print(
        "\n" + "-" * 70
    )


# ---------------------------------------------------------
# Show mismatches if any
# ---------------------------------------------------------

if length_mismatches:

    print("\n" + "=" * 70)
    print("LENGTH MISMATCH EXAMPLES")
    print("=" * 70)

    print(
        pd.DataFrame(
            length_mismatches
        ).head(20)
    )

In [ ]:
# =========================================================
# STANDARDIZATION
# =========================================================

import re
import pandas as pd


# ---------------------------------------------------------
# Text utilities
# ---------------------------------------------------------

def clean_text_value(value):
    """
    Convert a single value to clean text.
    """

    if value is None:
        return ""

    if isinstance(value, (list, tuple)):
        value = " ".join(
            str(x) for x in value
            if x is not None
        )

    value = str(value)

    value = re.sub(
        r"\s+",
        " ",
        value
    ).strip()

    return value


def normalize_text_for_split(text):
    """
    Normalize text for exact duplicate and leakage checks.
    """

    text = clean_text_value(text).lower()

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


# ---------------------------------------------------------
# PubMedQA
# ---------------------------------------------------------

def extract_pubmedqa_context(context_obj):
    """
    Concatenate all abstract context segments
    in their original order.
    """

    if not isinstance(context_obj, dict):
        return clean_text_value(
            context_obj
        )

    contexts = context_obj.get(
        "contexts",
        []
    )

    return clean_text_value(
        contexts
    )


pubmedqa_rows = []

for split_name in pubmedqa.keys():

    for row_idx, row in enumerate(
        pubmedqa[split_name]
    ):

        question = clean_text_value(
            row["question"]
        )

        passage = clean_text_value(
            extract_pubmedqa_context(
                row["context"]
            )
        )

        if not question or not passage:
            continue

        pubmedqa_rows.append({
            "source": "pubmedqa",
            "source_id": clean_text_value(
                row.get(
                    "pubid",
                    f"{split_name}_{row_idx}"
                )
            ),
            "original_split": split_name,
            "question": question,
            "passage": passage,
            "answer": "",
            "question_norm":
                normalize_text_for_split(
                    question
                ),
            "passage_norm":
                normalize_text_for_split(
                    passage
                )
        })


pubmedqa_std = pd.DataFrame(
    pubmedqa_rows
)


# ---------------------------------------------------------
# BioASQ
# ---------------------------------------------------------

def parse_bioasq_answer_context(text):
    """
    Extract answer metadata and retrieval context
    from the flattened BioASQ text field.
    """

    text = str(text)

    answer_match = re.search(
        r"<answer>\s*(.*?)\s*<context>",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    context_match = re.search(
        r"<context>\s*(.*)$",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    answer = (
        answer_match.group(1).strip()
        if answer_match
        else ""
    )

    context = (
        context_match.group(1).strip()
        if context_match
        else ""
    )

    return answer, context


bioasq_rows = []

for split_name in bioasq.keys():

    for row_idx, row in enumerate(
        bioasq[split_name]
    ):

        question = clean_text_value(
            row["question"]
        )

        answer, context = (
            parse_bioasq_answer_context(
                row["text"]
            )
        )

        answer = clean_text_value(
            answer
        )

        passage = clean_text_value(
            context
        )

        if not question or not passage:
            continue

        bioasq_rows.append({
            "source": "bioasq",
            "source_id":
                f"{split_name}_{row_idx}",
            "original_split":
                split_name,
            "question":
                question,
            "passage":
                passage,
            "answer":
                answer,
            "question_norm":
                normalize_text_for_split(
                    question
                ),
            "passage_norm":
                normalize_text_for_split(
                    passage
                )
        })


bioasq_std = pd.DataFrame(
    bioasq_rows
)


# ---------------------------------------------------------
# MedQuAD
# ---------------------------------------------------------

medquad_rows = []

for row_idx, row in medquad_raw.iterrows():

    question = clean_text_value(
        row["prompt"]
    )

    passage = clean_text_value(
        row["response"]
    )

    if not question or not passage:
        continue

    medquad_rows.append({
        "source": "medquad",
        "source_id":
            f"medquad_{row_idx}",
        "original_split":
            "unsplit",
        "question":
            question,
        "passage":
            passage,
        "answer":
            "",
        "question_norm":
            normalize_text_for_split(
                question
            ),
        "passage_norm":
            normalize_text_for_split(
                passage
            )
    })


medquad_std = pd.DataFrame(
    medquad_rows
)


# ---------------------------------------------------------
# Combine positive QA pairs
# ---------------------------------------------------------

all_pos_raw = pd.concat(
    [
        pubmedqa_std,
        bioasq_std,
        medquad_std
    ],
    ignore_index=True
)


all_pos = (
    all_pos_raw
    .drop_duplicates(
        subset=[
            "source",
            "question_norm",
            "passage_norm"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# Safety checks
# ---------------------------------------------------------

assert (
    bioasq_std["passage"]
    .str.contains(
        "<answer>",
        case=False,
        regex=False
    )
    .sum()
    == 0
)

assert (
    bioasq_std["passage"]
    .str.contains(
        "<context>",
        case=False,
        regex=False
    )
    .sum()
    == 0
)

assert (
    all_pos[
        [
            "source",
            "question_norm",
            "passage_norm"
        ]
    ]
    .duplicated()
    .sum()
    == 0
)


# ---------------------------------------------------------
# Report
# ---------------------------------------------------------

print("=" * 70)
print("STANDARDIZED POSITIVE DATA")
print("=" * 70)

print(
    "\nRaw standardized rows:"
)

print(
    "PubMedQA:",
    len(pubmedqa_std)
)

print(
    "BioASQ:",
    len(bioasq_std)
)

print(
    "MedQuAD:",
    len(medquad_std)
)

print(
    "\nPositive QA pairs after exact pair deduplication:"
)

print(
    all_pos["source"]
    .value_counts()
)

print(
    "\nTotal positive QA pairs:",
    len(all_pos)
)

print(
    "\nBioASQ unique questions:",
    bioasq_std[
        "question_norm"
    ].nunique()
)

print(
    "BioASQ unique context passages:",
    bioasq_std[
        "passage_norm"
    ].nunique()
)

print(
    "BioASQ answer-tag occurrences in retrieval passages:",
    bioasq_std[
        "passage"
    ]
    .str.contains(
        "<answer>",
        case=False,
        regex=False
    )
    .sum()
)

print(
    "\nOriginal split distribution:"
)

print(
    all_pos.groupby(
        [
            "source",
            "original_split"
        ]
    ).size()
)

print(
    "\nStandardization safety checks PASSED."
)

In [ ]:
print("=" * 70)
print("PUBMEDQA PASSAGE SANITY CHECK")
print("=" * 70)

print(pubmedqa_std.iloc[0]["question"])
print()
print(pubmedqa_std.iloc[0]["passage"][:1500])

print("\n" + "=" * 70)
print("BIOASQ PASSAGE SANITY CHECK")
print("=" * 70)

print(bioasq_std.iloc[0]["question"])
print()
print(bioasq_std.iloc[0]["passage"][:1500])
print("\nAnswer metadata:")
print(bioasq_std.iloc[0]["answer"])

print("\n" + "=" * 70)
print("MEDQUAD PASSAGE SANITY CHECK")
print("=" * 70)

print(medquad_std.iloc[0]["question"])
print()
print(medquad_std.iloc[0]["passage"][:1500])

assert "<answer>" not in bioasq_std.iloc[0]["passage"].lower()
assert "<context>" not in bioasq_std.iloc[0]["passage"].lower()

print("\nPassage sanity checks PASSED.")

In [ ]:
def word_len_series(s: pd.Series) -> pd.Series:
    return s.astype(str).str.split().apply(len)


def shorten(text, maxlen=160):
    text = str(text)
    return text if len(text) <= maxlen else text[:maxlen] + " ..."


def quick_stats(df: pd.DataFrame, name: str, show_examples=True, ex_n=2):
    qlen = word_len_series(df["question"])
    plen = word_len_series(df["passage"])

    stats = {
        "rows": len(df),
        "unique_questions": df["question_norm"].nunique(),
        "unique_passages": df["passage_norm"].nunique(),
        "avg_question_words": float(np.mean(qlen)),
        "avg_passage_words": float(np.mean(plen)),
        "median_question_words": float(np.median(qlen)),
        "median_passage_words": float(np.median(plen)),
        "max_question_words": int(np.max(qlen)) if len(qlen) else 0,
        "max_passage_words": int(np.max(plen)) if len(plen) else 0
    }

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    for key, value in stats.items():
        print(f"{key}: {value}")

    if show_examples:
        for i in range(min(ex_n, len(df))):
            row = df.iloc[i]
            print(f"\nExample {i + 1}")
            print("Q:", shorten(row["question"]))
            print("P:", shorten(row["passage"]))


quick_stats(pubmedqa_std, "PUBMEDQA")
quick_stats(bioasq_std, "BIOASQ")
quick_stats(medquad_std, "MEDQUAD")

In [ ]:
print("=" * 70)
print("BIOASQ DUPLICATION / MULTI-RELEVANCE AUDIT")
print("=" * 70)

unique_pairs = (
    bioasq_std[
        ["question_norm", "passage_norm"]
    ]
    .drop_duplicates()
)

print("Total standardized rows:", len(bioasq_std))
print("Unique questions:", bioasq_std["question_norm"].nunique())
print("Unique passages:", bioasq_std["passage_norm"].nunique())
print("Unique question-passage pairs:", len(unique_pairs))
print("Exact duplicate rows:", len(bioasq_std) - len(unique_pairs))

rows_per_question = (
    bioasq_std
    .groupby("question_norm")
    .size()
    .sort_values(ascending=False)
)

passages_per_question = (
    bioasq_std
    .groupby("question_norm")["passage_norm"]
    .nunique()
    .sort_values(ascending=False)
)

print("\nQuestions with more than one row:",
      int((rows_per_question > 1).sum()))

print("Maximum rows for one question:",
      int(rows_per_question.max()))

print("\nQuestions with more than one unique passage:",
      int((passages_per_question > 1).sum()))

print("Maximum unique passages for one question:",
      int(passages_per_question.max()))

print("\nTOP MULTI-PASSAGE QUESTIONS")

for q_norm in passages_per_question[
    passages_per_question > 1
].head(5).index:

    subset = bioasq_std[
        bioasq_std["question_norm"] == q_norm
    ]

    print("\n" + "-" * 70)
    print("QUESTION:")
    print(subset.iloc[0]["question"])
    print("Rows:", len(subset))
    print(
        "Unique passages:",
        subset["passage_norm"].nunique()
    )

    for i, passage in enumerate(
        subset["passage"].drop_duplicates().head(3),
        start=1
    ):
        print(f"\nPASSAGE {i}:")
        print(passage[:800])


bio_train = bioasq_std[
    bioasq_std["original_split"] == "train"
]

bio_val = bioasq_std[
    bioasq_std["original_split"] == "validation"
]

print("\n" + "=" * 70)
print("BIOASQ NATIVE TRAIN / VALIDATION OVERLAP")
print("=" * 70)

print(
    "Question overlap:",
    len(
        set(bio_train["question_norm"]) &
        set(bio_val["question_norm"])
    )
)

print(
    "Passage overlap:",
    len(
        set(bio_train["passage_norm"]) &
        set(bio_val["passage_norm"])
    )
)

train_pairs = set(
    zip(
        bio_train["question_norm"],
        bio_train["passage_norm"]
    )
)

val_pairs = set(
    zip(
        bio_val["question_norm"],
        bio_val["passage_norm"]
    )
)

print(
    "Exact question-passage pair overlap:",
    len(train_pairs & val_pairs)
)

In [ ]:
print("=" * 70)
print("POSITIVE DATASET SUMMARY")
print("=" * 70)

summary_rows = []

for source_name in [
    "pubmedqa",
    "bioasq",
    "medquad"
]:

    source_df = all_pos[
        all_pos["source"] == source_name
    ]

    relevant_per_question = (
        source_df
        .groupby("question_norm")["passage_norm"]
        .nunique()
    )

    summary_rows.append({
        "source": source_name,
        "positive_pairs": len(source_df),
        "unique_questions":
            source_df["question_norm"].nunique(),
        "unique_passages":
            source_df["passage_norm"].nunique(),
        "multi_relevant_questions":
            int((relevant_per_question > 1).sum()),
        "max_relevant_passages":
            int(relevant_per_question.max())
    })

dataset_summary = pd.DataFrame(summary_rows)
display(dataset_summary)

duplicate_pairs = all_pos.duplicated(
    subset=[
        "source",
        "question_norm",
        "passage_norm"
    ]
).sum()

print("\nGlobal positive pairs:", len(all_pos))
print(
    "Global unique questions:",
    all_pos["question_norm"].nunique()
)
print(
    "Duplicate source-question-passage pairs:",
    duplicate_pairs
)

assert duplicate_pairs == 0

print("\nPositive dataset structure check PASSED.")

## 3. Leakage-controlled train/validation/test partitions

Positive question-passage associations are partitioned before negative generation. Exact and near-duplicate controls are applied so that related items are not independently assigned across held-out partitions.

In [ ]:
# =========================================================
# NEAR-DUPLICATE-AWARE FINAL SPLIT
# Exact question/passage + TF-IDF similarity >= 0.95
# =========================================================

NEAR_THRESHOLD = 0.95
SPLIT_SEED = 42

df = all_pos.copy().reset_index(drop=True)
n = len(df)

# ---------------------------------------------------------
# Union-Find
# ---------------------------------------------------------

parent = np.arange(n)
rank = np.zeros(n, dtype=int)


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a, b):
    ra = find(a)
    rb = find(b)

    if ra == rb:
        return

    if rank[ra] < rank[rb]:
        parent[ra] = rb
    elif rank[ra] > rank[rb]:
        parent[rb] = ra
    else:
        parent[rb] = ra
        rank[ra] += 1


# ---------------------------------------------------------
# 1. Exact question connections
# ---------------------------------------------------------

for _, idxs in df.groupby("question_norm").groups.items():
    idxs = list(idxs)

    if len(idxs) > 1:
        base = idxs[0]

        for idx in idxs[1:]:
            union(base, idx)


# ---------------------------------------------------------
# 2. Exact passage connections
# ---------------------------------------------------------

for _, idxs in df.groupby("passage_norm").groups.items():
    idxs = list(idxs)

    if len(idxs) > 1:
        base = idxs[0]

        for idx in idxs[1:]:
            union(base, idx)


# ---------------------------------------------------------
# 3. Near-duplicate connections within each source
# ---------------------------------------------------------

def add_near_duplicate_edges(text_column, source_name):

    source_idx = df.index[
        df["source"] == source_name
    ].to_numpy()

    texts = (
        df.loc[source_idx, text_column]
        .fillna("")
        .astype(str)
        .tolist()
    )

    if len(texts) < 2:
        return 0

    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        max_features=50000
    )

    X = vectorizer.fit_transform(texts)

    nn = NearestNeighbors(
        metric="cosine",
        algorithm="brute",
        radius=1.0 - NEAR_THRESHOLD,
        n_jobs=-1
    )

    nn.fit(X)

    distances, neighbors = nn.radius_neighbors(
        X,
        return_distance=True
    )

    edge_count = 0

    for local_i, (dists, neighs) in enumerate(
        zip(distances, neighbors)
    ):

        global_i = source_idx[local_i]

        for dist, local_j in zip(dists, neighs):

            if local_j <= local_i:
                continue

            similarity = 1.0 - float(dist)

            if similarity >= NEAR_THRESHOLD:
                global_j = source_idx[local_j]

                union(global_i, global_j)
                edge_count += 1

    return edge_count


print("=" * 70)
print("ADDING NEAR-DUPLICATE CONNECTIONS")
print("=" * 70)

near_stats = []

for source_name in sorted(df["source"].unique()):

    q_edges = add_near_duplicate_edges(
        "question_norm",
        source_name
    )

    p_edges = add_near_duplicate_edges(
        "passage_norm",
        source_name
    )

    near_stats.append({
        "source": source_name,
        "question_edges": q_edges,
        "passage_edges": p_edges
    })

    print(
        source_name,
        "| question edges =", q_edges,
        "| passage edges =", p_edges
    )


# ---------------------------------------------------------
# 4. Build final components
# ---------------------------------------------------------

components_dict = defaultdict(list)

for i in range(n):
    components_dict[find(i)].append(i)

components = list(components_dict.values())

component_sizes = np.array(
    [len(c) for c in components]
)

print("\\n" + "=" * 70)
print("FINAL COMPONENT SUMMARY")
print("=" * 70)

print("Total rows:", n)
print("Total components:", len(components))
print("Largest component:", component_sizes.max())
print("Median component size:", np.median(component_sizes))
print("Components > 1 row:", np.sum(component_sizes > 1))


# ---------------------------------------------------------
# 5. Component-level 80 / 10 / 10 split
# ---------------------------------------------------------

rng = np.random.default_rng(SPLIT_SEED)

order = np.arange(len(components))
rng.shuffle(order)

targets = {
    "train": int(n * 0.80),
    "val": int(n * 0.10)
}

targets["test"] = (
    n - targets["train"] - targets["val"]
)

assigned = {
    "train": [],
    "val": [],
    "test": []
}

sizes = {
    "train": 0,
    "val": 0,
    "test": 0
}

for comp_id in order:

    comp = components[comp_id]
    comp_size = len(comp)

    remaining = {
        split: targets[split] - sizes[split]
        for split in sizes
    }

    chosen = max(
        remaining,
        key=remaining.get
    )

    assigned[chosen].extend(comp)
    sizes[chosen] += comp_size


train_pos = (
    df.iloc[assigned["train"]]
    .copy()
    .reset_index(drop=True)
)

val_pos = (
    df.iloc[assigned["val"]]
    .copy()
    .reset_index(drop=True)
)

test_pos = (
    df.iloc[assigned["test"]]
    .copy()
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 6. Final report
# ---------------------------------------------------------

print("\\n" + "=" * 70)
print("FINAL NEAR-DUPLICATE-AWARE SPLIT")
print("=" * 70)

print(f"Train: {len(train_pos):,}")
print(f"Val:   {len(val_pos):,}")
print(f"Test:  {len(test_pos):,}")

for split_name, split_data in [
    ("TRAIN", train_pos),
    ("VALIDATION", val_pos),
    ("TEST", test_pos)
]:
    print(f"\\n{split_name}")
    print(split_data["source"].value_counts())


# ---------------------------------------------------------
# 7. Exact overlap audit
# ---------------------------------------------------------

def overlap(a, b, col):
    return len(set(a[col]) & set(b[col]))


print("\\n" + "=" * 70)
print("FINAL EXACT OVERLAP AUDIT")
print("=" * 70)

exact_checks = {
    "Question Train-Val":
        overlap(train_pos, val_pos, "question_norm"),

    "Question Train-Test":
        overlap(train_pos, test_pos, "question_norm"),

    "Question Val-Test":
        overlap(val_pos, test_pos, "question_norm"),

    "Passage Train-Val":
        overlap(train_pos, val_pos, "passage_norm"),

    "Passage Train-Test":
        overlap(train_pos, test_pos, "passage_norm"),

    "Passage Val-Test":
        overlap(val_pos, test_pos, "passage_norm"),
}

for key, value in exact_checks.items():
    print(key, ":", value)

assert all(value == 0 for value in exact_checks.values())

print("\\nExact leakage check PASSED.")


# ---------------------------------------------------------
# 8. Source-specific final splits
# ---------------------------------------------------------

dataset_splits = {}

for source_name in sorted(df["source"].unique()):

    dataset_splits[source_name] = {

        "train":
            train_pos[
                train_pos["source"] == source_name
            ].reset_index(drop=True),

        "val":
            val_pos[
                val_pos["source"] == source_name
            ].reset_index(drop=True),

        "test":
            test_pos[
                test_pos["source"] == source_name
            ].reset_index(drop=True)
    }


print("\\n" + "=" * 70)
print("FINAL SOURCE-SPECIFIC SPLITS")
print("=" * 70)

for source_name, splits in dataset_splits.items():

    print(
        source_name,
        "| train =", len(splits["train"]),
        "| val =", len(splits["val"]),
        "| test =", len(splits["test"])
    )

In [ ]:
def show_exact_overlap_examples(
    df_a,
    df_b,
    column,
    name_a,
    name_b,
    max_examples=10
):
    common = set(df_a[column]) & set(df_b[column])

    print("\n" + "=" * 70)
    print(f"{name_a} vs {name_b} | {column}")
    print("Common values:", len(common))
    print("=" * 70)

    if not common:
        print("No exact overlap.")
        return

    for i, value in enumerate(list(common)[:max_examples], start=1):
        print(f"\n--- OVERLAP {i} ---")

        print(f"\n{name_a}:")
        display(
            df_a[df_a[column] == value][
                ["source", "question", "passage"]
            ]
        )

        print(f"\n{name_b}:")
        display(
            df_b[df_b[column] == value][
                ["source", "question", "passage"]
            ]
        )


for column in ["question_norm", "passage_norm"]:

    show_exact_overlap_examples(
        train_pos,
        val_pos,
        column,
        "TRAIN",
        "VALIDATION"
    )

    show_exact_overlap_examples(
        train_pos,
        test_pos,
        column,
        "TRAIN",
        "TEST"
    )

    show_exact_overlap_examples(
        val_pos,
        test_pos,
        column,
        "VALIDATION",
        "TEST"
    )

In [ ]:
NEAR_DUP_THRESHOLD = 0.95
MAX_FEATURES = 50000


def near_duplicate_audit(
    df_a,
    df_b,
    text_col,
    name_a,
    name_b,
    threshold=NEAR_DUP_THRESHOLD,
    top_examples=10
):

    texts_a = df_a[text_col].astype(str).tolist()
    texts_b = df_b[text_col].astype(str).tolist()

    all_texts = texts_a + texts_b

    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        max_features=MAX_FEATURES,
        min_df=1
    )

    X = vectorizer.fit_transform(all_texts)

    X_a = X[:len(texts_a)]
    X_b = X[len(texts_a):]

    sim = cosine_similarity(X_a, X_b)

    best_b_idx = sim.argmax(axis=1)
    best_scores = sim.max(axis=1)

    result = pd.DataFrame({
        "a_index": np.arange(len(df_a)),
        "b_index": best_b_idx,
        "similarity": best_scores
    })

    flagged = result[
        result["similarity"] >= threshold
    ].copy()

    flagged = flagged.sort_values(
        "similarity",
        ascending=False
    )

    print("\n" + "=" * 75)
    print(
        f"NEAR-DUPLICATE AUDIT: "
        f"{name_a} vs {name_b} | {text_col}"
    )
    print("=" * 75)

    print("Rows in A:", len(df_a))
    print("Rows in B:", len(df_b))
    print(
        f"Pairs with best-match similarity >= {threshold}:",
        len(flagged)
    )

    if len(flagged) == 0:
        print("No near-duplicates above threshold.")
        return flagged

    print("\nTop examples:")

    for _, row in flagged.head(top_examples).iterrows():

        a_idx = int(row["a_index"])
        b_idx = int(row["b_index"])

        print("\n" + "-" * 75)
        print("Similarity:", round(row["similarity"], 4))

        print(f"\n{name_a}:")
        print(df_a.iloc[a_idx][text_col])

        print(f"\n{name_b}:")
        print(df_b.iloc[b_idx][text_col])

    return flagged


q_train_val = near_duplicate_audit(
    train_pos,
    val_pos,
    "question_norm",
    "TRAIN",
    "VALIDATION"
)

q_train_test = near_duplicate_audit(
    train_pos,
    test_pos,
    "question_norm",
    "TRAIN",
    "TEST"
)

q_val_test = near_duplicate_audit(
    val_pos,
    test_pos,
    "question_norm",
    "VALIDATION",
    "TEST"
)

p_train_val = near_duplicate_audit(
    train_pos,
    val_pos,
    "passage_norm",
    "TRAIN",
    "VALIDATION"
)

p_train_test = near_duplicate_audit(
    train_pos,
    test_pos,
    "passage_norm",
    "TRAIN",
    "TEST"
)

p_val_test = near_duplicate_audit(
    val_pos,
    test_pos,
    "passage_norm",
    "VALIDATION",
    "TEST"
)

In [ ]:
def summarize_flagged_sources(
    flagged,
    df_a,
    df_b,
    label
):
    rows = []

    for _, row in flagged.iterrows():
        a_idx = int(row["a_index"])
        b_idx = int(row["b_index"])

        rows.append({
            "source_a": df_a.iloc[a_idx]["source"],
            "source_b": df_b.iloc[b_idx]["source"],
            "similarity": float(row["similarity"])
        })

    summary = pd.DataFrame(rows)

    print("\n" + "=" * 75)
    print(label)
    print("=" * 75)

    if len(summary) == 0:
        print("No flagged pairs.")
        return

    print("\nSource pairs:")
    print(
        summary.groupby(
            ["source_a", "source_b"]
        ).size().sort_values(ascending=False)
    )

    print("\nSimilarity distribution:")
    print(
        summary["similarity"].describe(
            percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
        )
    )


summarize_flagged_sources(
    q_train_val,
    train_pos,
    val_pos,
    "QUESTION: TRAIN vs VALIDATION"
)

summarize_flagged_sources(
    q_train_test,
    train_pos,
    test_pos,
    "QUESTION: TRAIN vs TEST"
)

summarize_flagged_sources(
    q_val_test,
    val_pos,
    test_pos,
    "QUESTION: VALIDATION vs TEST"
)

summarize_flagged_sources(
    p_train_val,
    train_pos,
    val_pos,
    "PASSAGE: TRAIN vs VALIDATION"
)

summarize_flagged_sources(
    p_train_test,
    train_pos,
    test_pos,
    "PASSAGE: TRAIN vs TEST"
)

summarize_flagged_sources(
    p_val_test,
    val_pos,
    test_pos,
    "PASSAGE: VALIDATION vs TEST"
)

In [ ]:
def save_split_with_hash(
    df,
    split_name
):

    output_path = os.path.join(
        SPLIT_DIR,
        f"{split_name}_positive.csv"
    )

    df.to_csv(
        output_path,
        index=False
    )

    with open(output_path, "rb") as f:
        sha256 = hashlib.sha256(
            f.read()
        ).hexdigest()

    return {
        "split": split_name,
        "rows": len(df),
        "unique_questions":
            df["question_norm"].nunique(),
        "unique_passages":
            df["passage_norm"].nunique(),
        "sha256": sha256,
        "file": output_path
    }


manifest = [
    save_split_with_hash(
        train_pos,
        "train"
    ),
    save_split_with_hash(
        val_pos,
        "validation"
    ),
    save_split_with_hash(
        test_pos,
        "test"
    )
]


manifest_data = {
    "seed": SEED,
    "split_seed": SPLIT_SEED,
    "near_duplicate_threshold":
        NEAR_THRESHOLD,
    "total_positive_pairs":
        len(all_pos),
    "splits":
        manifest
}


manifest_path = os.path.join(
    SPLIT_DIR,
    "split_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest_data,
        f,
        indent=2
    )


print("=" * 70)
print("FIXED POSITIVE SPLITS SAVED")
print("=" * 70)

for item in manifest:

    print(
        f"{item['split']:10s} | "
        f"rows = {item['rows']:5d} | "
        f"questions = "
        f"{item['unique_questions']:5d} | "
        f"passages = "
        f"{item['unique_passages']:5d}"
    )

    print(
        "SHA256:",
        item["sha256"]
    )


print("\nSaved directory:")
print(SPLIT_DIR)

print("\nManifest:")
print(manifest_path)

## 4. Structured training candidates and negative sampling

Each training instance contains one verified positive passage at candidate index 0, one random negative, and one TF-IDF hard negative. Known relevant passages for the same normalized question are excluded from negative sampling.

In [ ]:
# =========================================================
# TRAINING DATA STRUCTURE FOR CONTRASTIVE LEARNING
# =========================================================

# Contrastive training must use only verified positive
# question-passage pairs on the diagonal.
#
# Explicit negatives will be generated and passed separately
# in a later step; they must never appear as diagonal pairs.

train_positive_pairs = (
    train_pos[
        [
            "question",
            "passage",
            "question_norm",
            "passage_norm",
            "source",
            "source_id"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(train_positive_pairs) == len(train_pos)

assert train_positive_pairs[
    ["question_norm", "passage_norm"]
].duplicated().sum() == 0

print("=" * 70)
print("CONTRASTIVE TRAINING POSITIVE PAIRS")
print("=" * 70)

print("Rows:", len(train_positive_pairs))
print(
    "Unique questions:",
    train_positive_pairs["question_norm"].nunique()
)
print(
    "Unique passages:",
    train_positive_pairs["passage_norm"].nunique()
)

print("\nSource distribution:")
print(
    train_positive_pairs["source"].value_counts()
)

print(
    "\nDiagonal training pairs contain verified positives only."
)

In [ ]:
# =========================================================
# EXPLICIT NEGATIVE SAMPLING
# =========================================================

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors


NEGATIVE_SEED = 42

rng = np.random.default_rng(
    NEGATIVE_SEED
)


# ---------------------------------------------------------
# Validate required input
# ---------------------------------------------------------

required_columns = {
    "question",
    "passage",
    "question_norm",
    "passage_norm",
    "source",
    "source_id"
}

missing_columns = (
    required_columns -
    set(train_positive_pairs.columns)
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: "
        f"{sorted(missing_columns)}"
    )


# ---------------------------------------------------------
# Build known relevance sets
# ---------------------------------------------------------

known_relevant = (
    train_positive_pairs
    .groupby(
        [
            "source",
            "question_norm"
        ]
    )["passage_norm"]
    .apply(set)
    .to_dict()
)


# ---------------------------------------------------------
# Build source-specific passage pools
# ---------------------------------------------------------

source_passage_pools = {}

for source_name in sorted(
    train_positive_pairs["source"].unique()
):

    source_df = train_positive_pairs[
        train_positive_pairs["source"]
        == source_name
    ]

    passage_pool = (
        source_df[
            [
                "passage",
                "passage_norm"
            ]
        ]
        .drop_duplicates(
            subset=["passage_norm"]
        )
        .reset_index(drop=True)
    )

    if len(passage_pool) < 2:
        raise ValueError(
            f"Passage pool is too small for "
            f"source: {source_name}"
        )

    source_passage_pools[
        source_name
    ] = passage_pool


# ---------------------------------------------------------
# Build TF-IDF hard-negative indices
# ---------------------------------------------------------

tfidf_indices = {}

print("=" * 70)
print("BUILDING TF-IDF INDICES")
print("=" * 70)

for source_name, passage_pool in (
    source_passage_pools.items()
):

    print(
        f"{source_name}: "
        f"{len(passage_pool)} unique passages"
    )

    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        max_features=50000,
        min_df=1
    )

    passage_matrix = (
        vectorizer.fit_transform(
            passage_pool[
                "passage"
            ].astype(str)
        )
    )

    n_neighbors = min(
        50,
        len(passage_pool)
    )

    nn = NearestNeighbors(
        metric="cosine",
        algorithm="brute",
        n_neighbors=n_neighbors,
        n_jobs=-1
    )

    nn.fit(
        passage_matrix
    )

    tfidf_indices[
        source_name
    ] = {
        "vectorizer": vectorizer,
        "nn": nn,
        "n_neighbors": n_neighbors
    }


# ---------------------------------------------------------
# Sample negatives
# ---------------------------------------------------------

negative_rows = []

print("\n" + "=" * 70)
print("SAMPLING NEGATIVES")
print("=" * 70)

for _, row in tqdm(
    train_positive_pairs.iterrows(),
    total=len(train_positive_pairs),
    desc="Sampling negatives"
):

    source_name = row[
        "source"
    ]

    question = row[
        "question"
    ]

    question_norm = row[
        "question_norm"
    ]

    positive_passage = row[
        "passage"
    ]

    positive_passage_norm = row[
        "passage_norm"
    ]

    relevant_set = (
        known_relevant[
            (
                source_name,
                question_norm
            )
        ]
    )

    passage_pool = (
        source_passage_pools[
            source_name
        ]
    )


    # -----------------------------------------------------
    # Random negative
    # -----------------------------------------------------

    valid_random_mask = (
        ~passage_pool[
            "passage_norm"
        ].isin(
            relevant_set
        )
    )

    valid_random = (
        passage_pool[
            valid_random_mask
        ]
        .reset_index(drop=True)
    )

    if len(valid_random) == 0:
        raise ValueError(
            "No valid random negative "
            f"for source={source_name}, "
            f"question={question[:80]}"
        )

    random_idx = int(
        rng.integers(
            low=0,
            high=len(valid_random)
        )
    )

    random_candidate = (
        valid_random.iloc[
            random_idx
        ]
    )

    random_negative = (
        random_candidate[
            "passage"
        ]
    )

    random_negative_norm = (
        random_candidate[
            "passage_norm"
        ]
    )


    # -----------------------------------------------------
    # TF-IDF hard negative
    # -----------------------------------------------------

    index_data = (
        tfidf_indices[
            source_name
        ]
    )

    query_vector = (
        index_data[
            "vectorizer"
        ].transform(
            [str(question)]
        )
    )

    distances, neighbor_indices = (
        index_data[
            "nn"
        ].kneighbors(
            query_vector,
            n_neighbors=index_data[
                "n_neighbors"
            ],
            return_distance=True
        )
    )

    hard_negative = None
    hard_negative_norm = None
    hard_similarity = None

    for (
        distance,
        candidate_idx
    ) in zip(
        distances[0],
        neighbor_indices[0]
    ):

        candidate = (
            passage_pool.iloc[
                int(candidate_idx)
            ]
        )

        candidate_norm = (
            candidate[
                "passage_norm"
            ]
        )

        if (
            candidate_norm
            in relevant_set
        ):
            continue

        if (
            candidate_norm
            == random_negative_norm
        ):
            continue

        hard_negative = (
            candidate[
                "passage"
            ]
        )

        hard_negative_norm = (
            candidate_norm
        )

        hard_similarity = (
            1.0 -
            float(distance)
        )

        break


    # Fallback if top-50 contains no valid hard negative
    if hard_negative is None:

        valid_hard = (
            passage_pool[
                ~passage_pool[
                    "passage_norm"
                ].isin(
                    relevant_set
                )
            ]
        )

        valid_hard = (
            valid_hard[
                valid_hard[
                    "passage_norm"
                ]
                != random_negative_norm
            ]
            .reset_index(drop=True)
        )

        if len(valid_hard) == 0:
            raise ValueError(
                "No valid hard negative "
                f"for source={source_name}, "
                f"question={question[:80]}"
            )

        fallback_idx = int(
            rng.integers(
                low=0,
                high=len(valid_hard)
            )
        )

        fallback_candidate = (
            valid_hard.iloc[
                fallback_idx
            ]
        )

        hard_negative = (
            fallback_candidate[
                "passage"
            ]
        )

        hard_negative_norm = (
            fallback_candidate[
                "passage_norm"
            ]
        )

        hard_similarity = np.nan


    # -----------------------------------------------------
    # Store one structured training example
    # -----------------------------------------------------

    negative_rows.append({
        "source":
            source_name,

        "source_id":
            row["source_id"],

        "question":
            question,

        "question_norm":
            question_norm,

        "positive_passage":
            positive_passage,

        "positive_passage_norm":
            positive_passage_norm,

        "random_negative":
            random_negative,

        "random_negative_norm":
            random_negative_norm,

        "hard_negative":
            hard_negative,

        "hard_negative_norm":
            hard_negative_norm,

        "hard_negative_similarity":
            hard_similarity
    })


train_with_negatives = (
    pd.DataFrame(
        negative_rows
    )
)


# ---------------------------------------------------------
# Safety checks
# ---------------------------------------------------------

invalid_random = 0
invalid_hard = 0
same_random_hard = 0

for _, row in train_with_negatives.iterrows():

    relevant_set = (
        known_relevant[
            (
                row["source"],
                row["question_norm"]
            )
        ]
    )

    if (
        row[
            "random_negative_norm"
        ]
        in relevant_set
    ):
        invalid_random += 1

    if (
        row[
            "hard_negative_norm"
        ]
        in relevant_set
    ):
        invalid_hard += 1

    if (
        row[
            "random_negative_norm"
        ]
        ==
        row[
            "hard_negative_norm"
        ]
    ):
        same_random_hard += 1


assert invalid_random == 0
assert invalid_hard == 0
assert same_random_hard == 0

assert (
    len(train_with_negatives)
    ==
    len(train_positive_pairs)
)


# ---------------------------------------------------------
# Final report
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("EXPLICIT NEGATIVE SAMPLING COMPLETE")
print("=" * 70)

print(
    "Training examples:",
    len(train_with_negatives)
)

print(
    "Random negatives:",
    train_with_negatives[
        "random_negative"
    ].notna().sum()
)

print(
    "TF-IDF hard negatives:",
    train_with_negatives[
        "hard_negative"
    ].notna().sum()
)

print(
    "Relevant random negatives:",
    invalid_random
)

print(
    "Relevant hard negatives:",
    invalid_hard
)

print(
    "Duplicate random/hard negatives:",
    same_random_hard
)

print("\nSource distribution:")
print(
    train_with_negatives[
        "source"
    ].value_counts()
)

print(
    "\nTF-IDF hard-negative "
    "similarity distribution:"
)

print(
    train_with_negatives[
        "hard_negative_similarity"
    ].describe()
)

print(
    "\nNegative sampling safety "
    "checks PASSED."
)

In [ ]:
# =========================================================
# SAVE STRUCTURED CONTRASTIVE TRAINING DATA
# =========================================================

os.makedirs(DATA_DIR, exist_ok=True)

negative_output_path = os.path.join(
    DATA_DIR,
    "train_contrastive_with_explicit_negatives.csv"
)

train_with_negatives.to_csv(
    negative_output_path,
    index=False
)

with open(negative_output_path, "rb") as f:
    negative_sha256 = hashlib.sha256(f.read()).hexdigest()

print("=" * 70)
print("STRUCTURED CONTRASTIVE TRAINING DATA SAVED")
print("=" * 70)
print("Path:", negative_output_path)
print("Rows:", len(train_with_negatives))
print("SHA256:", negative_sha256)

## 5. Model registry and training configuration

All encoders use attention-mask-aware mean pooling in this implementation. E5 query/passage prefixes are applied by the tokenizer helper. The ranking and distillation temperatures are fixed at 0.05, and the teacher remains frozen during KD.

In [ ]:
# =========================================================
# 1. MOUNT GOOGLE DRIVE + IMPORTS + DEVICE
# =========================================================

import os
import json
import math
import time
import random
import hashlib
import gc
import shutil
from pathlib import Path
from dataclasses import dataclass, asdict

from google.colab import drive

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.stats import spearmanr

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup


# Use a dedicated empty mount point.
# This avoids the "Mountpoint must not already contain files" error.

In [ ]:
ROOT = Path(PROJECT_DIR)
RUN_DIR = ROOT / "runs"
RUN_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "train_neg": Path(DATA_DIR) / "train_contrastive_with_explicit_negatives.csv",
    "train_pos": Path(SPLIT_DIR) / "train_positive.csv",
    "val_pos": Path(SPLIT_DIR) / "validation_positive.csv",
    "test_pos": Path(SPLIT_DIR) / "test_positive.csv",
}

for name, path in FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required file {name}: {path}. Run Sections 2-4 first.")

train_neg = pd.read_csv(FILES["train_neg"])
train_pos = pd.read_csv(FILES["train_pos"])
val_pos = pd.read_csv(FILES["val_pos"])
test_pos = pd.read_csv(FILES["test_pos"])
print({k: str(v) for k, v in FILES.items()})

In [ ]:
# =========================================================
# 4. MODEL REGISTRY + GLOBAL CONFIG
# =========================================================
MODEL_REGISTRY = {
    "T1_PubMedBERT": {"name":"microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract", "pooling":"mean", "e5":False},
    "T2_MiniLM": {"name":"sentence-transformers/all-MiniLM-L6-v2", "pooling":"mean", "e5":False},
    "T3_E5base": {"name":"intfloat/e5-base", "pooling":"mean", "e5":True},
    "S1_DistilBERT": {"name":"distilbert-base-uncased", "pooling":"mean", "e5":False},
    "S2_BioBERT": {"name":"dmis-lab/biobert-base-cased-v1.1", "pooling":"mean", "e5":False},
    "S3_MiniLM": {"name":"sentence-transformers/all-MiniLM-L6-v2", "pooling":"mean", "e5":False},
}

@dataclass
class Config:
    max_len: int = 256
    batch_size: int = 32
    eval_batch_size: int = 64
    epochs: int = 5
    lr: float = 3e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.10
    tau_rank: float = 0.05
    tau_teacher: float = 0.05
    tau_student: float = 0.05
    lambda_kd: float = 1.0
    alpha: float = 0.5
    grad_clip: float = 1.0
    geometry_top_n: int = 100
    seed: int = SEED

CFG = Config()
print(json.dumps(asdict(CFG), indent=2))
print(json.dumps(MODEL_REGISTRY, indent=2))

In [ ]:
# =========================================================
# 5. SAFE TOKENIZER + BI-ENCODER
# =========================================================
def load_tokenizer(name):
    try:
        return AutoTokenizer.from_pretrained(name, use_fast=True)
    except Exception:
        return AutoTokenizer.from_pretrained(name, use_fast=False)

def mean_pool(last_hidden, mask):
    mask = mask.unsqueeze(-1).to(last_hidden.dtype)
    return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-6)

class BiEncoder(nn.Module):
    def __init__(self, spec):
        super().__init__()
        self.spec = spec
        self.encoder = AutoModel.from_pretrained(spec["name"])
        self.out_dim = int(self.encoder.config.hidden_size)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        if self.spec["pooling"] == "mean":
            z = mean_pool(out.last_hidden_state, attention_mask)
        elif self.spec["pooling"] == "cls":
            z = out.last_hidden_state[:,0]
        else:
            raise ValueError("Unknown pooling: " + str(self.spec["pooling"]))
        return F.normalize(z, p=2, dim=-1)

def format_texts(texts, spec, kind):
    texts = [str(x) for x in texts]
    if spec.get("e5", False):
        prefix = "query: " if kind == "query" else "passage: "
        texts = [prefix + x for x in texts]
    return texts

def tokenize_texts(tok, texts, spec, kind, max_len):
    texts = format_texts(texts, spec, kind)
    return tok(texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt")

### Implementation fidelity notes

The code below intentionally preserves the implementation used for the reported experiments. After attention-mask-aware mean pooling, encoder outputs are L2-normalized. The embedding-distillation term therefore aligns normalized teacher embeddings with L2-normalized projected student embeddings. In the implementation, the query and positive-passage MSE terms are averaged with a factor of 0.5. Ranking logits use cosine similarity with `tau_rank=0.05`; teacher and student KD distributions are then formed from the corresponding cosine scores using `tau_teacher=0.05` and `tau_student=0.05`.

The main K3 teacher-student comparison uses `alpha=0.50`. The controlled five-seed K3 robustness experiment uses `alpha=0.10`, selected from the validation sweep. Every run executes all five training epochs and retains the checkpoint with the highest validation macro MRR@10; training is not terminated before epoch 5.


In [ ]:
# =========================================================
# 6. STRUCTURED TRAINING DATASET
# =========================================================
class CandidateDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return {
            "source": str(r["source"]),
            "question": str(r["question"]),
            "positive": str(r["positive_passage"]),
            "random_negative": str(r["random_negative"]),
            "hard_negative": str(r["hard_negative"]),
        }

def collate_candidates(batch):
    return {k:[x[k] for x in batch] for k in batch[0]}

def make_train_loader(df, batch_size, shuffle=True):
    g = torch.Generator(); g.manual_seed(CFG.seed)
    return DataLoader(CandidateDataset(df), batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=True, collate_fn=collate_candidates, generator=g)

train_loader = make_train_loader(train_neg, CFG.batch_size)
first = next(iter(train_loader))
print("Batch size:", len(first["question"]))
print("Candidate fields OK:", list(first.keys()))

In [ ]:
# =========================================================
# 7. LOSS FUNCTIONS
# =========================================================
def candidate_logits(model, tok, spec, batch, max_len, tau):
    B = len(batch["question"])
    q_enc = tokenize_texts(tok, batch["question"], spec, "query", max_len)
    candidates = []
    for p, r, h in zip(batch["positive"], batch["random_negative"], batch["hard_negative"]):
        candidates.extend([p, r, h])
    p_enc = tokenize_texts(tok, candidates, spec, "passage", max_len)
    q_ids, q_mask = q_enc["input_ids"].to(DEVICE), q_enc["attention_mask"].to(DEVICE)
    p_ids, p_mask = p_enc["input_ids"].to(DEVICE), p_enc["attention_mask"].to(DEVICE)
    q = model(q_ids, q_mask)
    p = model(p_ids, p_mask).view(B, 3, -1)
    logits = torch.einsum("bd,bcd->bc", q, p) / tau
    return logits, q, p

def rank_loss(student_logits):
    labels = torch.zeros(student_logits.size(0), dtype=torch.long, device=student_logits.device)
    return F.cross_entropy(student_logits, labels)

def teacher_rank_kd(student_logits, teacher_logits, tau_s, tau_t):
    # logits are cosine/tau_rank; convert back to cosine-like scores before KD temperatures
    s = student_logits * CFG.tau_rank / max(tau_s, 1e-8)
    t = teacher_logits * CFG.tau_rank / max(tau_t, 1e-8)
    t_prob = F.softmax(t, dim=-1)
    s_log = F.log_softmax(s, dim=-1)
    return F.kl_div(s_log, t_prob, reduction="batchmean") * (tau_s ** 2)

def embedding_kd(q_s, p_s, q_t, p_t, projector):
    q_sp = F.normalize(projector(q_s), p=2, dim=-1)
    p_sp = F.normalize(projector(p_s[:,0,:]), p=2, dim=-1)
    return 0.5 * (F.mse_loss(q_sp, q_t) + F.mse_loss(p_sp, p_t[:,0,:]))

In [ ]:
# =========================================================
# 8. SMOKE TEST: FORWARD/BACKWARD + TEACHER ISOLATION
# =========================================================
def smoke_test(teacher_key="T1_PubMedBERT", student_key="S1_DistilBERT"):
    t_spec, s_spec = MODEL_REGISTRY[teacher_key], MODEL_REGISTRY[student_key]
    t_tok, s_tok = load_tokenizer(t_spec["name"]), load_tokenizer(s_spec["name"])
    teacher, student = BiEncoder(t_spec).to(DEVICE), BiEncoder(s_spec).to(DEVICE)
    for p in teacher.parameters(): p.requires_grad = False
    teacher.eval()
    projector = nn.Linear(student.out_dim, teacher.out_dim, bias=False).to(DEVICE)
    batch = next(iter(make_train_loader(train_neg.head(8), 4, shuffle=False)))
    with torch.no_grad():
        t_logits, q_t, p_t = candidate_logits(teacher,t_tok,t_spec,batch,CFG.max_len,CFG.tau_rank)
    s_logits, q_s, p_s = candidate_logits(student,s_tok,s_spec,batch,CFG.max_len,CFG.tau_rank)
    l_rank = rank_loss(s_logits)
    l_emb = embedding_kd(q_s,p_s,q_t,p_t,projector)
    l_kl = teacher_rank_kd(s_logits,t_logits,CFG.tau_student,CFG.tau_teacher)
    loss = l_rank + CFG.lambda_kd*(CFG.alpha*l_emb + (1-CFG.alpha)*l_kl)
    assert torch.isfinite(loss), f"Non-finite smoke loss: {loss.item()}"
    loss.backward()
    assert any(p.grad is not None for p in student.parameters() if p.requires_grad)
    assert all(p.grad is None for p in teacher.parameters())
    assert projector.weight.grad is not None
    print({"rank":float(l_rank),"embed":float(l_emb),"teacher_rank":float(l_kl),"total":float(loss)})
    print("Smoke test PASSED: forward/backward finite; teacher gradients isolated.")
    del teacher, student, projector; gc.collect(); torch.cuda.empty_cache()

smoke_test()

In [ ]:
# =========================================================
# 9. MULTI-RELEVANCE RETRIEVAL EVALUATOR
# =========================================================
@torch.no_grad()
def encode_unique(model, tok, spec, texts, kind, batch_size=64):
    model.eval(); out=[]
    texts=list(texts)
    for i in range(0,len(texts),batch_size):
        enc=tokenize_texts(tok,texts[i:i+batch_size],spec,kind,CFG.max_len)
        z=model(enc["input_ids"].to(DEVICE),enc["attention_mask"].to(DEVICE))
        out.append(z.cpu())
    return torch.cat(out,0)

def build_eval_source(df, source):
    d=df[df.source==source].copy()
    qtab=d[["question_norm","question"]].drop_duplicates("question_norm").reset_index(drop=True)
    ptab=d[["passage_norm","passage"]].drop_duplicates("passage_norm").reset_index(drop=True)
    pidx={p:i for i,p in enumerate(ptab.passage_norm)}
    rel=(d.groupby("question_norm")["passage_norm"].apply(lambda x:{pidx[v] for v in x}).to_dict())
    return qtab,ptab,rel

@torch.no_grad()
def evaluate_retrieval(model,tok,spec,df,ks=(1,5,10)):
    rows=[]
    for source in sorted(df.source.unique()):
        qtab,ptab,rel=build_eval_source(df,source)
        Q=encode_unique(model,tok,spec,qtab.question.tolist(),"query",CFG.eval_batch_size)
        P=encode_unique(model,tok,spec,ptab.passage.tolist(),"passage",CFG.eval_batch_size)
        sims=Q@P.T
        metrics={f"mrr@{k}":[] for k in ks}
        metrics.update({f"hit@{k}":[] for k in ks})
        metrics.update({f"recall@{k}":[] for k in ks})
        for qi,qn in enumerate(qtab.question_norm):
            order=torch.argsort(sims[qi],descending=True).tolist(); gold=rel[qn]
            ranks=[r+1 for r,p in enumerate(order) if p in gold]
            first=min(ranks)
            for k in ks:
                metrics[f"mrr@{k}"].append(1.0/first if first<=k else 0.0)
                metrics[f"hit@{k}"].append(float(first<=k))
                metrics[f"recall@{k}"].append(sum(r<=k for r in ranks)/len(gold))
        row={"source":source,"questions":len(qtab),"candidates":len(ptab)}
        row.update({m:float(np.mean(v)) for m,v in metrics.items()})
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
# =========================================================
# 10. RANKING-GEOMETRY EVALUATOR
# =========================================================
@torch.no_grad()
def evaluate_geometry(student,s_tok,s_spec,teacher,t_tok,t_spec,df,top_n=None):
    top_n = top_n or CFG.geometry_top_n
    out=[]
    for source in sorted(df.source.unique()):
        qtab,ptab,_=build_eval_source(df,source)
        SQ=encode_unique(student,s_tok,s_spec,qtab.question.tolist(),"query",CFG.eval_batch_size)
        SP=encode_unique(student,s_tok,s_spec,ptab.passage.tolist(),"passage",CFG.eval_batch_size)
        TQ=encode_unique(teacher,t_tok,t_spec,qtab.question.tolist(),"query",CFG.eval_batch_size)
        TP=encode_unique(teacher,t_tok,t_spec,ptab.passage.tolist(),"passage",CFG.eval_batch_size)
        S=SQ@SP.T; T=TQ@TP.T
        rhos=[]; agreements=[]
        for i in range(len(qtab)):
            n=min(top_n,T.shape[1]); idx=torch.topk(T[i],k=n).indices
            ts=T[i,idx].numpy(); ss=S[i,idx].numpy()
            rho=spearmanr(ts,ss).statistic
            if np.isfinite(rho): rhos.append(float(rho))
            # exact pairwise ordering agreement within teacher top-n
            dt=ts[:,None]-ts[None,:]; ds=ss[:,None]-ss[None,:]
            tri=np.triu_indices(n,k=1); valid=dt[tri]!=0
            if valid.any(): agreements.append(float(np.mean(np.sign(dt[tri][valid])==np.sign(ds[tri][valid]))))
        out.append({"source":source,"questions":len(qtab),"geometry_top_n":top_n,
                    "spearman_mean":float(np.mean(rhos)),"pairwise_order_agreement":float(np.mean(agreements))})
    return pd.DataFrame(out)

In [ ]:
# =========================================================
# 11. TRAIN ONE RUN: NoKD / TEACHER / K1 / K2 / K3
# =========================================================
def train_one_run(student_key, strategy="NoKD", teacher_key=None, alpha=None, epochs=None, run_tag=None):
    assert strategy in {"NoKD","TEACHER","K1","K2","K3"}
    if strategy in {"K1","K2","K3"}: assert teacher_key is not None
    alpha = CFG.alpha if alpha is None else float(alpha)
    epochs = CFG.epochs if epochs is None else int(epochs)
    if strategy=="K3": assert 0.0 <= alpha <= 1.0

    # TEACHER means task-finetune the selected teacher architecture with no external teacher.
    model_key = teacher_key if strategy=="TEACHER" else student_key
    spec=MODEL_REGISTRY[model_key]; tok=load_tokenizer(spec["name"]); model=BiEncoder(spec).to(DEVICE)

    teacher=t_tok=t_spec=projector=None
    if strategy in {"K1","K2","K3"}:
        t_spec=MODEL_REGISTRY[teacher_key]; t_tok=load_tokenizer(t_spec["name"]); teacher=BiEncoder(t_spec).to(DEVICE)
        for p in teacher.parameters(): p.requires_grad=False
        teacher.eval()
        projector=nn.Linear(model.out_dim,teacher.out_dim,bias=False).to(DEVICE)

    params=list(model.parameters()) + ([] if projector is None else list(projector.parameters()))
    opt=torch.optim.AdamW(params,lr=CFG.lr,weight_decay=CFG.weight_decay)
    loader=make_train_loader(train_neg,CFG.batch_size,shuffle=True)
    total_steps=epochs*len(loader); warm=int(total_steps*CFG.warmup_ratio)
    sched=get_linear_schedule_with_warmup(opt,warm,total_steps)

    stamp=time.strftime("%Y%m%d_%H%M%S")
    run_name=run_tag or f"{strategy}_{model_key}" + (f"_{teacher_key}" if teacher_key else "") + (f"_a{alpha:.2f}" if strategy=="K3" else "") + f"_s{CFG.seed}_{stamp}"
    outdir=RUN_DIR/run_name; outdir.mkdir(parents=True,exist_ok=True)
    config={**asdict(CFG),"strategy":strategy,"student_key":student_key,"teacher_key":teacher_key,"model_key":model_key,"alpha":alpha,"epochs":epochs}
    json.dump(config,open(outdir/"config.json","w"),indent=2)

    history=[]; best=-1.0
    for ep in range(1,epochs+1):
        model.train(); losses=[]
        for batch in tqdm(loader,desc=f"{run_name} epoch {ep}/{epochs}"):
            opt.zero_grad(set_to_none=True)
            s_logits,q_s,p_s=candidate_logits(model,tok,spec,batch,CFG.max_len,CFG.tau_rank)
            l_rank=rank_loss(s_logits); l_emb=torch.tensor(0.,device=DEVICE); l_kl=torch.tensor(0.,device=DEVICE)
            if teacher is not None:
                with torch.no_grad(): t_logits,q_t,p_t=candidate_logits(teacher,t_tok,t_spec,batch,CFG.max_len,CFG.tau_rank)
                if strategy in {"K1","K3"}: l_emb=embedding_kd(q_s,p_s,q_t,p_t,projector)
                if strategy in {"K2","K3"}: l_kl=teacher_rank_kd(s_logits,t_logits,CFG.tau_student,CFG.tau_teacher)
            if strategy in {"NoKD","TEACHER"}: loss=l_rank
            elif strategy=="K1": loss=l_rank+CFG.lambda_kd*l_emb
            elif strategy=="K2": loss=l_rank+CFG.lambda_kd*l_kl
            else: loss=l_rank+CFG.lambda_kd*(alpha*l_emb+(1-alpha)*l_kl)
            if not torch.isfinite(loss): raise FloatingPointError(f"Non-finite loss in {run_name}")
            loss.backward(); torch.nn.utils.clip_grad_norm_(params,CFG.grad_clip); opt.step(); sched.step(); losses.append(float(loss.detach()))
        val=evaluate_retrieval(model,tok,spec,val_pos)
        macro=float(val["mrr@10"].mean())
        rec={"epoch":ep,"train_loss":float(np.mean(losses)),"val_macro_mrr@10":macro}; history.append(rec)
        pd.DataFrame(history).to_csv(outdir/"history.csv",index=False)
        torch.save({"model":model.state_dict(),"projector":None if projector is None else projector.state_dict(),"config":config},outdir/"last.pt")
        if macro>best:
            best=macro; torch.save({"model":model.state_dict(),"projector":None if projector is None else projector.state_dict(),"config":config},outdir/"best.pt")
        print(rec); display(val)

    ck=torch.load(outdir/"best.pt",map_location=DEVICE); model.load_state_dict(ck["model"])
    test=evaluate_retrieval(model,tok,spec,test_pos); test.to_csv(outdir/"test_metrics.csv",index=False)
    print("BEST VAL macro MRR@10:",best); display(test)
    return {"model":model,"tokenizer":tok,"spec":spec,"teacher":teacher,"teacher_tokenizer":t_tok,"teacher_spec":t_spec,"outdir":str(outdir),"test":test}

In [ ]:
# =========================================================
# 12. QUICK TRAINING SMOKE RUN (1 MINI-EPOCH / 64 ROWS)
# Does not modify the frozen files.
# =========================================================
def mini_training_smoke():
    small=train_neg.head(64).copy()
    spec=MODEL_REGISTRY["S1_DistilBERT"]; tok=load_tokenizer(spec["name"]); model=BiEncoder(spec).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=CFG.lr)
    loader=make_train_loader(small,8,shuffle=False)
    model.train(); vals=[]
    for batch in loader:
        opt.zero_grad(set_to_none=True)
        logits,_,_=candidate_logits(model,tok,spec,batch,CFG.max_len,CFG.tau_rank)
        loss=rank_loss(logits); assert torch.isfinite(loss); loss.backward(); opt.step(); vals.append(float(loss))
    print("Mini training smoke PASSED. Mean loss:",np.mean(vals))
    del model; gc.collect(); torch.cuda.empty_cache()
mini_training_smoke()

## 6. Main experiments

The main comparison spans the three teacher encoders and three student encoders. `RUN_MAIN_EXPERIMENTS` is disabled by default because the full matrix is computationally expensive. Enable it to reproduce the complete training matrix. Validation macro MRR@10 selects the checkpoint subsequently evaluated on the test split.

In [ ]:
TEACHERS = ["T1_PubMedBERT", "T2_MiniLM", "T3_E5base"]
STUDENTS = ["S1_DistilBERT", "S2_BioBERT", "S3_MiniLM"]
STRATEGIES = ["K1", "K2", "K3"]
MAIN_K3_ALPHA = 0.50
RUN_MAIN_EXPERIMENTS = False

main_results = []
if RUN_MAIN_EXPERIMENTS:
    for teacher_key in TEACHERS:
        for student_key in STUDENTS:
            for strategy in STRATEGIES:
                alpha = MAIN_K3_ALPHA if strategy == "K3" else None
                result = train_one_run(student_key, strategy=strategy, teacher_key=teacher_key, alpha=alpha)
                table = result["test"].copy()
                table["teacher"] = teacher_key
                table["student"] = student_key
                table["strategy"] = strategy
                main_results.append(table)
    pd.concat(main_results, ignore_index=True).to_csv(RESULTS_DIR + "/main_kd_results.csv", index=False)
else:
    print("Main 3x3x3 experiment matrix is disabled. Set RUN_MAIN_EXPERIMENTS=True to execute it.")

### 6.1 Teacher-only, task-fine-tuned teacher, and No-KD controls

In [ ]:
RUN_BASELINES = False
baseline_results = {}

@torch.no_grad()
def evaluate_pretrained_teacher(model_key):
    spec = MODEL_REGISTRY[model_key]
    tok = load_tokenizer(spec["name"])
    model = BiEncoder(spec).to(DEVICE)
    metrics = evaluate_retrieval(model, tok, spec, test_pos)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return metrics

if RUN_BASELINES:
    # Direct retrieval with the original pretrained teacher encoders.
    for teacher_key in TEACHERS:
        teacher_metrics = evaluate_pretrained_teacher(teacher_key)
        teacher_metrics.to_csv(Path(RESULTS_DIR) / f"pretrained_{teacher_key}_test_metrics.csv", index=False)
        baseline_results[f"Pretrained_{teacher_key}"] = teacher_metrics

    # Teacher-free student controls trained with L_rank only.
    for student_key in STUDENTS:
        baseline_results[f"NoKD_{student_key}"] = train_one_run(student_key, strategy="NoKD")

    # Task-fine-tuned PubMedBERT control.
    baseline_results["T1_FT"] = train_one_run(
        "S1_DistilBERT", strategy="TEACHER", teacher_key="T1_PubMedBERT"
    )
else:
    print("Baselines are disabled. Set RUN_BASELINES=True to execute them.")


## 7. K3 alpha sensitivity analysis

The controlled sweep uses PubMedBERT as teacher and DistilBERT as student, seed 42, and includes both single-component endpoints.

In [ ]:
RUN_ALPHA_SWEEP = False
ALPHAS = [0.00, 0.10, 0.30, 0.50, 0.70, 0.90, 1.00]
alpha_results = {}
if RUN_ALPHA_SWEEP:
    set_global_seed(42) if "set_global_seed" in globals() else None
    CFG.seed = 42
    for alpha in ALPHAS:
        alpha_results[alpha] = train_one_run(
            "S1_DistilBERT", strategy="K3", teacher_key="T1_PubMedBERT", alpha=alpha,
            run_tag=f"alpha_sweep_{alpha:.2f}"
        )
else:
    print("Alpha sweep is disabled. Set RUN_ALPHA_SWEEP=True to execute it.")

## 8. Five-seed robustness experiments

The repeated-seed protocol uses seeds 42, 52, 62, 72, and 82 for the controlled PubMedBERT-to-DistilBERT setting. K3 uses alpha=0.10, selected by validation MRR@10 in the controlled alpha sweep. No-KD, K1, K2, and K3 are evaluated under matched seeds.

In [ ]:
# =========================================================
# 8. MULTI-SEED REPRODUCIBILITY UTILITY
# =========================================================

def set_global_seed(seed):
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    CFG.seed = seed
    return seed

set_global_seed(42)
print("Seed utility ready. Current seed:", CFG.seed)

In [ ]:
# =========================================================
# 11. RESTART-SAFE MULTI-SEED TRAINING
# =========================================================

MULTISEED_ROOT = ROOT / "multiseed_runs"
MULTISEED_ROOT.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 52, 62, 72, 82]

EXPERIMENTS = [
    {"label": "NoKD", "strategy": "NoKD", "student_key": "S1_DistilBERT", "teacher_key": None, "alpha": None},
    {
        "label": "K1",
        "strategy": "K1",
        "student_key": "S1_DistilBERT",
        "teacher_key": "T1_PubMedBERT",
        "alpha": None,
    },
    {
        "label": "K2",
        "strategy": "K2",
        "student_key": "S1_DistilBERT",
        "teacher_key": "T1_PubMedBERT",
        "alpha": None,
    },
    {
        "label": "K3_alpha010",
        "strategy": "K3",
        "student_key": "S1_DistilBERT",
        "teacher_key": "T1_PubMedBERT",
        "alpha": 0.10,
    },
]


def atomic_json_dump(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)


def run_complete(run_dir):
    run_dir = Path(run_dir)
    required = [
        run_dir / "COMPLETE.json",
        run_dir / "config.json",
        run_dir / "history.csv",
        run_dir / "test_metrics.csv",
        run_dir / "test_per_query.csv",
    ]
    return all(p.exists() for p in required)


def config_matches(cfg, exp, seed):
    if not isinstance(cfg, dict):
        return False

    checks = [
        cfg.get("strategy") == exp["strategy"],
        cfg.get("student_key") == exp["student_key"],
        cfg.get("teacher_key") == exp["teacher_key"],
        int(cfg.get("seed", -1)) == int(seed),
        int(cfg.get("epochs", -1)) == int(CFG.epochs),
    ]

    if exp["strategy"] == "K3":
        try:
            checks.append(
                abs(float(cfg.get("alpha")) - float(exp["alpha"])) < 1e-12
            )
        except Exception:
            return False

    return all(checks)


def find_compatible_stageD_checkpoint(exp, seed):
    """
    Reuse a previous compatible best checkpoint only if its config exactly
    matches the fixed multi-seed protocol. This is especially useful for
    seed 42. If no compatible checkpoint exists, return None.
    """
    candidates = []

    if not RUN_DIR.exists():
        return None

    for config_path in RUN_DIR.rglob("config.json"):
        try:
            cfg = json.load(open(config_path))
        except Exception:
            continue

        if not config_matches(cfg, exp, seed):
            continue

        best_pt = config_path.parent / "best.pt"
        history_csv = config_path.parent / "history.csv"

        if best_pt.exists() and history_csv.exists():
            candidates.append(config_path.parent)

    if not candidates:
        return None

    # Prefer most recently modified compatible run.
    candidates.sort(
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )
    return candidates[0]


def load_model_from_best_checkpoint(run_dir, exp):
    run_dir = Path(run_dir)
    spec = MODEL_REGISTRY[exp["student_key"]]
    tok = load_tokenizer(spec["name"])
    model = BiEncoder(spec).to(DEVICE)

    ck = torch.load(
        run_dir / "best.pt",
        map_location=DEVICE
    )
    model.load_state_dict(ck["model"])
    model.eval()

    return model, tok, spec


def import_compatible_run(source_run_dir, target_run_dir, exp, seed):
    """
    Imports a compatible old best checkpoint into the new result structure,
    then recomputes test aggregate + per-query metrics with the current
    corrected evaluator. No training is performed.
    """
    source_run_dir = Path(source_run_dir)
    target_run_dir = Path(target_run_dir)
    target_run_dir.mkdir(parents=True, exist_ok=True)

    print("Compatible previous compatible run found:")
    print(source_run_dir)
    print("Reusing checkpoint; training will NOT be repeated.")

    model, tok, spec = load_model_from_best_checkpoint(
        source_run_dir, exp
    )

    history = pd.read_csv(source_run_dir / "history.csv")
    best_idx = history["val_macro_mrr@10"].idxmax()
    best_epoch = int(history.loc[best_idx, "epoch"])
    best_val = float(history.loc[best_idx, "val_macro_mrr@10"])

    test, per_query = evaluate_retrieval_with_queries(
        model, tok, spec, test_pos
    )

    config = {
        **asdict(CFG),
        "seed": int(seed),
        "strategy": exp["strategy"],
        "student_key": exp["student_key"],
        "teacher_key": exp["teacher_key"],
        "alpha": exp["alpha"],
        "epochs": CFG.epochs,
        "selection_metric": "validation macro MRR@10",
        "imported_from": str(source_run_dir),
    }

    history.to_csv(target_run_dir / "history.csv", index=False)
    test.to_csv(target_run_dir / "test_metrics.csv", index=False)
    per_query.to_csv(target_run_dir / "test_per_query.csv", index=False)

    atomic_json_dump(
        config,
        target_run_dir / "config.json"
    )

    atomic_json_dump(
        {
            "status": "complete",
            "seed": int(seed),
            "strategy": exp["strategy"],
            "best_epoch": best_epoch,
            "best_val_macro_mrr@10": best_val,
            "source": "compatible_stageD_checkpoint",
        },
        target_run_dir / "COMPLETE.json"
    )

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "status": "IMPORTED",
        "best_epoch": best_epoch,
        "best_val": best_val,
    }


def train_multiseed_run(exp, seed):
    set_global_seed(seed)

    run_dir = (
        MULTISEED_ROOT /
        exp["label"] /
        f"seed_{seed}"
    )
    run_dir.mkdir(parents=True, exist_ok=True)

    if run_complete(run_dir):
        print(
            f"[SKIP] {exp['label']} seed={seed} "
            "already complete."
        )
        return {"status": "SKIPPED"}

    # Reuse an exact compatible old checkpoint when possible.
    old_run = find_compatible_stageD_checkpoint(exp, seed)

    if old_run is not None:
        return import_compatible_run(
            old_run,
            run_dir,
            exp,
            seed
        )

    print("\n" + "=" * 80)
    print(f"RUNNING {exp['label']} | seed={seed}")
    print("=" * 80)

    strategy = exp["strategy"]
    student_key = exp["student_key"]
    teacher_key = exp["teacher_key"]
    alpha = (
        CFG.alpha
        if exp["alpha"] is None
        else float(exp["alpha"])
    )

    # Same model/loss definitions as the main training section.
    spec = MODEL_REGISTRY[student_key]
    tok = load_tokenizer(spec["name"])
    model = BiEncoder(spec).to(DEVICE)

    t_spec = MODEL_REGISTRY[teacher_key]
    t_tok = load_tokenizer(t_spec["name"])
    teacher = BiEncoder(t_spec).to(DEVICE)

    for p in teacher.parameters():
        p.requires_grad = False
    teacher.eval()

    projector = nn.Linear(
        model.out_dim,
        teacher.out_dim,
        bias=False
    ).to(DEVICE)

    params = list(model.parameters()) + list(projector.parameters())

    opt = torch.optim.AdamW(
        params,
        lr=CFG.lr,
        weight_decay=CFG.weight_decay
    )

    # Created after setting the run-specific seed.
    loader = make_train_loader(
        train_neg,
        CFG.batch_size,
        shuffle=True
    )

    total_steps = CFG.epochs * len(loader)
    warm = int(total_steps * CFG.warmup_ratio)

    sched = get_linear_schedule_with_warmup(
        opt,
        warm,
        total_steps
    )

    config = {
        **asdict(CFG),
        "seed": int(seed),
        "strategy": strategy,
        "student_key": student_key,
        "teacher_key": teacher_key,
        "model_key": student_key,
        "alpha": (
            float(exp["alpha"])
            if exp["alpha"] is not None
            else None
        ),
        "epochs": CFG.epochs,
        "selection_metric": "validation macro MRR@10",
        "frozen_train_sha256": EXPECTED_SHA256["train"],
        "frozen_validation_sha256": EXPECTED_SHA256["validation"],
        "frozen_test_sha256": EXPECTED_SHA256["test"],
        "frozen_train_neg_sha256": EXPECTED_SHA256["train_neg"],
    }

    atomic_json_dump(
        config,
        run_dir / "config.json"
    )

    history = []
    best = -1.0
    best_epoch = None

    for ep in range(1, CFG.epochs + 1):
        model.train()
        losses = []

        for batch in tqdm(
            loader,
            desc=f"{exp['label']} s{seed} epoch {ep}/{CFG.epochs}"
        ):
            opt.zero_grad(set_to_none=True)

            s_logits, q_s, p_s = candidate_logits(
                model,
                tok,
                spec,
                batch,
                CFG.max_len,
                CFG.tau_rank
            )

            l_rank = rank_loss(s_logits)
            l_emb = torch.tensor(0.0, device=DEVICE)
            l_kl = torch.tensor(0.0, device=DEVICE)

            with torch.no_grad():
                t_logits, q_t, p_t = candidate_logits(
                    teacher,
                    t_tok,
                    t_spec,
                    batch,
                    CFG.max_len,
                    CFG.tau_rank
                )

            if strategy in {"K1", "K3"}:
                l_emb = embedding_kd(
                    q_s, p_s,
                    q_t, p_t,
                    projector
                )

            if strategy in {"K2", "K3"}:
                l_kl = teacher_rank_kd(
                    s_logits,
                    t_logits,
                    CFG.tau_student,
                    CFG.tau_teacher
                )

            if strategy == "K1":
                loss = (
                    l_rank +
                    CFG.lambda_kd * l_emb
                )

            elif strategy == "K2":
                loss = (
                    l_rank +
                    CFG.lambda_kd * l_kl
                )

            elif strategy == "K3":
                loss = (
                    l_rank +
                    CFG.lambda_kd * (
                        alpha * l_emb +
                        (1.0 - alpha) * l_kl
                    )
                )

            else:
                raise ValueError(strategy)

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"Non-finite loss in "
                    f"{exp['label']} seed={seed}"
                )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                params,
                CFG.grad_clip
            )
            opt.step()
            sched.step()

            losses.append(
                float(loss.detach())
            )

        val = evaluate_retrieval(
            model,
            tok,
            spec,
            val_pos
        )

        macro = float(
            val["mrr@10"].mean()
        )

        rec = {
            "epoch": ep,
            "train_loss": float(np.mean(losses)),
            "val_macro_mrr@10": macro,
        }
        history.append(rec)

        pd.DataFrame(history).to_csv(
            run_dir / "history.csv",
            index=False
        )

        # Save validation table for every epoch.
        val.to_csv(
            run_dir / f"val_metrics_epoch_{ep}.csv",
            index=False
        )

        torch.save(
            {
                "model": model.state_dict(),
                "projector": projector.state_dict(),
                "config": config,
            },
            run_dir / "last.pt"
        )

        if macro > best:
            best = macro
            best_epoch = ep

            torch.save(
                {
                    "model": model.state_dict(),
                    "projector": projector.state_dict(),
                    "config": config,
                },
                run_dir / "best.pt"
            )

        print(rec)
        display(val)

    # Final test evaluation only from validation-selected best checkpoint.
    ck = torch.load(
        run_dir / "best.pt",
        map_location=DEVICE
    )
    model.load_state_dict(
        ck["model"]
    )

    test, per_query = evaluate_retrieval_with_queries(
        model,
        tok,
        spec,
        test_pos
    )

    test.to_csv(
        run_dir / "test_metrics.csv",
        index=False
    )

    per_query.to_csv(
        run_dir / "test_per_query.csv",
        index=False
    )

    atomic_json_dump(
        {
            "status": "complete",
            "seed": int(seed),
            "strategy": strategy,
            "alpha": (
                float(exp["alpha"])
                if exp["alpha"] is not None
                else None
            ),
            "best_epoch": int(best_epoch),
            "best_val_macro_mrr@10": float(best),
            "completed_at": time.strftime(
                "%Y-%m-%d %H:%M:%S"
            ),
        },
        run_dir / "COMPLETE.json"
    )

    print(
        f"\nCOMPLETE: {exp['label']} seed={seed}"
    )
    print(
        "BEST VAL macro MRR@10:",
        best
    )
    display(test)

    # Free GPU memory before next run.
    del model, teacher, projector
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "status": "TRAINED",
        "best_epoch": best_epoch,
        "best_val": best,
    }


print("Multi-seed training engine ready.")
print("Output root:", MULTISEED_ROOT)

In [ ]:
# =========================================================
# 12. PREFLIGHT: SHOW EXACT RUN PLAN
# =========================================================

print("=" * 80)
print("MULTI-SEED RUN PLAN")
print("=" * 80)

print("Seeds:", SEEDS)
print("K3 alpha: 0.10")
print("Selection metric: validation macro MRR@10")
print("Frozen split hashes: VERIFIED")

for exp in EXPERIMENTS:
    print("\n", exp["label"])
    for seed in SEEDS:
        run_dir = (
            MULTISEED_ROOT /
            exp["label"] /
            f"seed_{seed}"
        )

        if run_complete(run_dir):
            state = "SKIP: already complete"
        else:
            old = find_compatible_stageD_checkpoint(
                exp, seed
            )
            if old is not None:
                state = f"IMPORT compatible checkpoint: {old.name}"
            else:
                state = "TRAIN"

        print(f"  seed {seed}: {state}")

print("\nPreflight complete.")

In [ ]:
# =========================================================
# 13. RUN ALL MISSING MULTI-SEED EXPERIMENTS
# =========================================================
# Safe to rerun after a disconnect.
# Completed runs are skipped automatically.

RUN_MULTISEED = False

run_status = []

if RUN_MULTISEED:
    for exp in EXPERIMENTS:
        for seed in SEEDS:
            result = train_multiseed_run(
                exp,
                seed
            )

            run_status.append({
                "experiment": exp["label"],
                "seed": seed,
                **result,
            })

            pd.DataFrame(run_status).to_csv(
                MULTISEED_ROOT /
                "current_run_status.csv",
                index=False
            )

    print("\n" + "=" * 80)
    print("ALL REQUESTED MULTI-SEED RUNS PROCESSED")
    print("=" * 80)
else:
    print("RUN_MULTISEED=False; no training started.")

In [ ]:
# =========================================================
# 14. AGGREGATE MULTI-SEED RESULTS
#     mean ± SD + 95% CI
# =========================================================

SUMMARY_DIR = MULTISEED_ROOT / "summaries"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

metric_cols = [
    "mrr@1", "mrr@5", "mrr@10",
    "hit@1", "hit@5", "hit@10",
    "recall@1", "recall@5", "recall@10",
]

all_rows = []
validation_rows = []

for exp in EXPERIMENTS:
    for seed in SEEDS:
        run_dir = (
            MULTISEED_ROOT /
            exp["label"] /
            f"seed_{seed}"
        )

        if not run_complete(run_dir):
            print(
                "INCOMPLETE:",
                exp["label"],
                seed
            )
            continue

        test = pd.read_csv(
            run_dir / "test_metrics.csv"
        )

        complete = json.load(
            open(run_dir / "COMPLETE.json")
        )

        validation_rows.append({
            "experiment": exp["label"],
            "seed": seed,
            "best_epoch": complete["best_epoch"],
            "best_val_macro_mrr@10":
                complete["best_val_macro_mrr@10"],
        })

        for row in test.to_dict("records"):
            row["experiment"] = exp["label"]
            row["seed"] = seed
            all_rows.append(row)

        # Macro across the three sources, matching main model-selection protocol.
        macro = {
            "experiment": exp["label"],
            "seed": seed,
            "source": "MACRO",
            "questions": int(test["questions"].sum()),
            "candidates": int(test["candidates"].sum()),
        }

        for m in metric_cols:
            macro[m] = float(test[m].mean())

        all_rows.append(macro)

results_long = pd.DataFrame(all_rows)
validation_df = pd.DataFrame(validation_rows)

results_long.to_csv(
    SUMMARY_DIR / "all_seed_test_metrics.csv",
    index=False
)

validation_df.to_csv(
    SUMMARY_DIR / "all_seed_validation_selection.csv",
    index=False
)


def summarize_group(g):
    out = {}

    n = len(g)
    out["n_seeds"] = n

    for m in metric_cols:
        vals = g[m].astype(float)

        mean = vals.mean()
        sd = vals.std(ddof=1) if n > 1 else np.nan
        sem = sd / np.sqrt(n) if n > 1 else np.nan

        if n > 1:
            critical = t.ppf(
                0.975,
                df=n - 1
            )
            half = critical * sem
            lo = mean - half
            hi = mean + half
        else:
            lo = np.nan
            hi = np.nan

        out[f"{m}_mean"] = mean
        out[f"{m}_sd"] = sd
        out[f"{m}_ci95_low"] = lo
        out[f"{m}_ci95_high"] = hi

    return pd.Series(out)


summary = (
    results_long
    .groupby(
        ["experiment", "source"],
        as_index=False
    )
    .apply(
        summarize_group,
        include_groups=False
    )
    .reset_index()
)

# Handle pandas versions where group keys may be represented differently.
if "level_0" in summary.columns:
    summary = summary.drop(
        columns=[
            c for c in ["level_0", "level_1"]
            if c in summary.columns
        ]
    )

summary.to_csv(
    SUMMARY_DIR / "multiseed_mean_sd_ci95.csv",
    index=False
)

print("=" * 80)
print("MULTI-SEED SUMMARY")
print("=" * 80)

display(
    results_long[
        results_long["source"] == "MACRO"
    ].sort_values(
        ["experiment", "seed"]
    )
)

print("\nMean ± SD / 95% CI:")
display(summary)

print("\nSaved to:")
print(SUMMARY_DIR)

In [ ]:
# =========================================================
# 15. ANALYSIS-READY COMPACT TABLE + COMPLETENESS AUDIT
# =========================================================

expected = len(EXPERIMENTS) * len(SEEDS)
completed = 0

for exp in EXPERIMENTS:
    for seed in SEEDS:
        run_dir = (
            MULTISEED_ROOT /
            exp["label"] /
            f"seed_{seed}"
        )
        completed += int(
            run_complete(run_dir)
        )

print("Completed runs:", completed, "/", expected)

if completed != expected:
    print(
        "Not all runs are complete yet. "
        "Rerun Cell 13 after reconnecting; "
        "completed runs will be skipped."
    )
else:
    macro_df = results_long[
        results_long["source"] == "MACRO"
    ].copy()

    compact_rows = []

    for exp_name, g in macro_df.groupby("experiment"):
        row = {
            "experiment": exp_name,
            "n_seeds": len(g),
        }

        for m in ["mrr@1", "mrr@5", "mrr@10",
                  "recall@1", "recall@5", "recall@10"]:
            mean = g[m].mean()
            sd = g[m].std(ddof=1)
            row[f"{m}_mean"] = mean
            row[f"{m}_sd"] = sd
            row[f"{m}_mean±sd"] = f"{mean:.4f} ± {sd:.4f}"

        compact_rows.append(row)

    analysis_table = pd.DataFrame(
        compact_rows
    ).sort_values("experiment")

    analysis_table.to_csv(
        SUMMARY_DIR /
        "analysis_ready_multiseed_table.csv",
        index=False
    )

    display(analysis_table)

    print("\nMULTI-SEED ROBUSTNESS EXPERIMENT COMPLETE.")
    print(
        "Per-query files are preserved for the "
        "next paired-bootstrap/randomization step."
    )

## 9. Ranking-order correspondence analysis

Ranking geometry is operationalized as teacher-student candidate-order correspondence. Spearman rank correlation and exact pairwise ordering agreement are computed over the teacher-defined top-N=100 candidates for each query. This analysis is separate from retrieval effectiveness.

In [ ]:
def run_ranking_geometry_analysis(student_bundle, teacher_bundle, evaluation_df, top_n=100):
    old_top_n = CFG.geometry_top_n
    CFG.geometry_top_n = top_n
    try:
        return evaluate_geometry(
            student_bundle["model"], student_bundle["tokenizer"], student_bundle["spec"],
            teacher_bundle["model"], teacher_bundle["tokenizer"], teacher_bundle["spec"],
            evaluation_df, top_n=top_n
        )
    finally:
        CFG.geometry_top_n = old_top_n

print("Ranking-order analysis helper ready (top-N=100).")

## 10. Loss-scale and gradient-norm audit

This audit quantifies the unweighted loss magnitudes and the corresponding student gradient norms for the K3 components. It is intended to show that equal alpha coefficients do not imply equal numerical or optimization influence.

In [ ]:
# =========================================================
# 4. MODEL REGISTRY + GLOBAL CONFIG
# =========================================================
MODEL_REGISTRY = {
    "T1_PubMedBERT": {"name":"microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract", "pooling":"mean", "e5":False},
    "T2_MiniLM": {"name":"sentence-transformers/all-MiniLM-L6-v2", "pooling":"mean", "e5":False},
    "T3_E5base": {"name":"intfloat/e5-base", "pooling":"mean", "e5":True},
    "S1_DistilBERT": {"name":"distilbert-base-uncased", "pooling":"mean", "e5":False},
    "S2_BioBERT": {"name":"dmis-lab/biobert-base-cased-v1.1", "pooling":"mean", "e5":False},
    "S3_MiniLM": {"name":"sentence-transformers/all-MiniLM-L6-v2", "pooling":"mean", "e5":False},
}

@dataclass
class Config:
    max_len: int = 256
    batch_size: int = 32
    eval_batch_size: int = 64
    epochs: int = 5
    lr: float = 3e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.10
    tau_rank: float = 0.05
    tau_teacher: float = 0.05
    tau_student: float = 0.05
    lambda_kd: float = 1.0
    alpha: float = 0.5
    grad_clip: float = 1.0
    geometry_top_n: int = 100
    seed: int = SEED

CFG = Config()
print(json.dumps(asdict(CFG), indent=2))
print(json.dumps(MODEL_REGISTRY, indent=2))

In [ ]:
# =========================================================
# 5. SAFE TOKENIZER + BI-ENCODER
# =========================================================
def load_tokenizer(name):
    try:
        return AutoTokenizer.from_pretrained(name, use_fast=True)
    except Exception:
        return AutoTokenizer.from_pretrained(name, use_fast=False)

def mean_pool(last_hidden, mask):
    mask = mask.unsqueeze(-1).to(last_hidden.dtype)
    return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-6)

class BiEncoder(nn.Module):
    def __init__(self, spec):
        super().__init__()
        self.spec = spec
        self.encoder = AutoModel.from_pretrained(spec["name"])
        self.out_dim = int(self.encoder.config.hidden_size)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        if self.spec["pooling"] == "mean":
            z = mean_pool(out.last_hidden_state, attention_mask)
        elif self.spec["pooling"] == "cls":
            z = out.last_hidden_state[:,0]
        else:
            raise ValueError("Unknown pooling: " + str(self.spec["pooling"]))
        return F.normalize(z, p=2, dim=-1)

def format_texts(texts, spec, kind):
    texts = [str(x) for x in texts]
    if spec.get("e5", False):
        prefix = "query: " if kind == "query" else "passage: "
        texts = [prefix + x for x in texts]
    return texts

def tokenize_texts(tok, texts, spec, kind, max_len):
    texts = format_texts(texts, spec, kind)
    return tok(texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt")

In [ ]:
# =========================================================
# 6. STRUCTURED TRAINING DATASET
# =========================================================
class CandidateDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return {
            "source": str(r["source"]),
            "question": str(r["question"]),
            "positive": str(r["positive_passage"]),
            "random_negative": str(r["random_negative"]),
            "hard_negative": str(r["hard_negative"]),
        }

def collate_candidates(batch):
    return {k:[x[k] for x in batch] for k in batch[0]}

def make_train_loader(df, batch_size, shuffle=True):
    g = torch.Generator(); g.manual_seed(CFG.seed)
    return DataLoader(CandidateDataset(df), batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=True, collate_fn=collate_candidates, generator=g)

train_loader = make_train_loader(train_neg, CFG.batch_size)
first = next(iter(train_loader))
print("Batch size:", len(first["question"]))
print("Candidate fields OK:", list(first.keys()))

In [ ]:
# =========================================================
# 7. LOSS FUNCTIONS
# =========================================================
def candidate_logits(model, tok, spec, batch, max_len, tau):
    B = len(batch["question"])
    q_enc = tokenize_texts(tok, batch["question"], spec, "query", max_len)
    candidates = []
    for p, r, h in zip(batch["positive"], batch["random_negative"], batch["hard_negative"]):
        candidates.extend([p, r, h])
    p_enc = tokenize_texts(tok, candidates, spec, "passage", max_len)
    q_ids, q_mask = q_enc["input_ids"].to(DEVICE), q_enc["attention_mask"].to(DEVICE)
    p_ids, p_mask = p_enc["input_ids"].to(DEVICE), p_enc["attention_mask"].to(DEVICE)
    q = model(q_ids, q_mask)
    p = model(p_ids, p_mask).view(B, 3, -1)
    logits = torch.einsum("bd,bcd->bc", q, p) / tau
    return logits, q, p

def rank_loss(student_logits):
    labels = torch.zeros(student_logits.size(0), dtype=torch.long, device=student_logits.device)
    return F.cross_entropy(student_logits, labels)

def teacher_rank_kd(student_logits, teacher_logits, tau_s, tau_t):
    # logits are cosine/tau_rank; convert back to cosine-like scores before KD temperatures
    s = student_logits * CFG.tau_rank / max(tau_s, 1e-8)
    t = teacher_logits * CFG.tau_rank / max(tau_t, 1e-8)
    t_prob = F.softmax(t, dim=-1)
    s_log = F.log_softmax(s, dim=-1)
    return F.kl_div(s_log, t_prob, reduction="batchmean") * (tau_s ** 2)

def embedding_kd(q_s, p_s, q_t, p_t, projector):
    q_sp = F.normalize(projector(q_s), p=2, dim=-1)
    p_sp = F.normalize(projector(p_s[:,0,:]), p=2, dim=-1)
    return 0.5 * (F.mse_loss(q_sp, q_t) + F.mse_loss(p_sp, p_t[:,0,:]))

In [ ]:

# =========================================================
# ANALYSIS AUDIT: LOSS SCALE + COMPONENT GRADIENT NORMS
# NO RETRAINING / NO OPTIMIZER STEP
# =========================================================

from pathlib import Path
import math, json, gc, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

AUDIT_ROOT = ROOT / "analysis_gradient_audit"
AUDIT_ROOT.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 1. Locate an existing K3 checkpoint.
# Preference: multiseed K3 seed 42; fallback: Stage-D K3.
# ---------------------------------------------------------
def _config_of(run_dir):
    p = run_dir / "config.json"
    if not p.exists():
        return None
    try:
        return json.load(open(p))
    except Exception:
        return None

candidates = []
search_roots = [ROOT / "multiseed_runs" / "K3", ROOT / "stageD_runs"]

for sr in search_roots:
    if not sr.exists():
        continue
    for ckpt in sr.rglob("best.pt"):
        cfg = _config_of(ckpt.parent)
        if not cfg or cfg.get("strategy") != "K3":
            continue
        # This audit is for the manuscript's main T1 -> S1 setting.
        if cfg.get("student_key") != "S1_DistilBERT":
            continue
        if cfg.get("teacher_key") != "T1_PubMedBERT":
            continue
        candidates.append((ckpt, cfg))

if not candidates:
    raise FileNotFoundError(
        "No existing K3 S1_DistilBERT <- T1_PubMedBERT best.pt was found. "
        "No training was started. Confirm that the completed K3 run directory is visible."
    )

# Prefer seed 42 and alpha 0.5 when available; otherwise use the first compatible run.
def pref(item):
    ck, cfg = item
    return (
        int(cfg.get("seed") == 42),
        int(abs(float(cfg.get("alpha", 0.5)) - 0.5) < 1e-12),
        ck.stat().st_mtime
    )

candidates = sorted(candidates, key=pref, reverse=True)
CKPT_PATH, RUN_CFG = candidates[0]

print("Selected existing checkpoint:")
print(CKPT_PATH)
print(json.dumps({
    "strategy": RUN_CFG.get("strategy"),
    "student_key": RUN_CFG.get("student_key"),
    "teacher_key": RUN_CFG.get("teacher_key"),
    "alpha": RUN_CFG.get("alpha"),
    "seed": RUN_CFG.get("seed"),
}, indent=2))

# ---------------------------------------------------------
# 2. Reconstruct the exact Stage-D model objects and load
#    the already-trained student/projector checkpoint.
#    Teacher is reconstructed exactly as in Stage-D K3:
#    frozen pretrained teacher, no optimizer/update.
# ---------------------------------------------------------
student_key = RUN_CFG["student_key"]
teacher_key = RUN_CFG["teacher_key"]
alpha = float(RUN_CFG.get("alpha", 0.5))
lambda_kd = float(RUN_CFG.get("lambda_kd", CFG.lambda_kd))

s_spec = MODEL_REGISTRY[student_key]
t_spec = MODEL_REGISTRY[teacher_key]

s_tok = load_tokenizer(s_spec["name"])
t_tok = load_tokenizer(t_spec["name"])

student = BiEncoder(s_spec).to(DEVICE)
teacher = BiEncoder(t_spec).to(DEVICE)
projector = nn.Linear(student.out_dim, teacher.out_dim, bias=False).to(DEVICE)

ck = torch.load(CKPT_PATH, map_location=DEVICE)
student.load_state_dict(ck["model"], strict=True)
if ck.get("projector") is None:
    raise RuntimeError("Selected K3 checkpoint has no saved projector state.")
projector.load_state_dict(ck["projector"], strict=True)

for p in teacher.parameters():
    p.requires_grad = False
teacher.eval()
student.eval()
projector.eval()

print(f"student_dim={student.out_dim} | teacher_dim={teacher.out_dim}")
print("Checkpoint loaded. No optimizer is created.")

# ---------------------------------------------------------
# 3. Helpers: compute global L2 gradient norm without
#    modifying model parameters.
# ---------------------------------------------------------
student_params = [p for p in student.parameters() if p.requires_grad]
projector_params = [p for p in projector.parameters() if p.requires_grad]

def global_grad_norm(grads):
    sq = 0.0
    used = 0
    for g in grads:
        if g is not None:
            sq += float(g.detach().float().pow(2).sum().cpu())
            used += 1
    return math.sqrt(sq), used

def grad_norm_of(loss, params, retain_graph=True):
    grads = torch.autograd.grad(
        loss, params, retain_graph=retain_graph,
        create_graph=False, allow_unused=True
    )
    return global_grad_norm(grads)

# ---------------------------------------------------------
# 4. Audit fixed batches only.
#    No shuffle and no optimizer.step().
# ---------------------------------------------------------
AUDIT_BATCHES = 10
AUDIT_BATCH_SIZE = min(int(RUN_CFG.get("batch_size", CFG.batch_size)), 32)

loader = make_train_loader(train_neg, AUDIT_BATCH_SIZE, shuffle=False)
rows = []

for batch_idx, batch in enumerate(loader):
    if batch_idx >= AUDIT_BATCHES:
        break

    # Teacher signal: no gradient.
    with torch.no_grad():
        t_logits, q_t, p_t = candidate_logits(
            teacher, t_tok, t_spec, batch,
            int(RUN_CFG.get("max_len", CFG.max_len)),
            float(RUN_CFG.get("tau_rank", CFG.tau_rank))
        )

    # Student graph shared by all three components.
    s_logits, q_s, p_s = candidate_logits(
        student, s_tok, s_spec, batch,
        int(RUN_CFG.get("max_len", CFG.max_len)),
        float(RUN_CFG.get("tau_rank", CFG.tau_rank))
    )

    l_rank = rank_loss(s_logits)
    l_embed = embedding_kd(q_s, p_s, q_t, p_t, projector)
    l_teacher_rank = teacher_rank_kd(
        s_logits, t_logits,
        float(RUN_CFG.get("tau_student", CFG.tau_student)),
        float(RUN_CFG.get("tau_teacher", CFG.tau_teacher))
    )

    # Unweighted component gradient norms on the shared student.
    g_rank_s, _ = grad_norm_of(l_rank, student_params, retain_graph=True)
    g_embed_s, _ = grad_norm_of(l_embed, student_params, retain_graph=True)
    g_trank_s, _ = grad_norm_of(l_teacher_rank, student_params, retain_graph=True)

    # Projector participates only in embedding KD.
    g_embed_p, _ = grad_norm_of(l_embed, projector_params, retain_graph=True)

    # Actual K3 weighted KD branches.
    weighted_embed = lambda_kd * alpha * l_embed
    weighted_trank = lambda_kd * (1.0 - alpha) * l_teacher_rank

    g_wembed_s, _ = grad_norm_of(weighted_embed, student_params, retain_graph=True)
    g_wtrank_s, _ = grad_norm_of(weighted_trank, student_params, retain_graph=True)
    g_wembed_p, _ = grad_norm_of(weighted_embed, projector_params, retain_graph=True)

    total = l_rank + weighted_embed + weighted_trank

    rows.append({
        "batch": batch_idx,
        "n": len(batch["question"]),
        "alpha": alpha,
        "lambda_kd": lambda_kd,
        "L_rank": float(l_rank.detach().cpu()),
        "L_embed": float(l_embed.detach().cpu()),
        "L_teacher_rank": float(l_teacher_rank.detach().cpu()),
        "weighted_embed_loss": float(weighted_embed.detach().cpu()),
        "weighted_teacher_rank_loss": float(weighted_trank.detach().cpu()),
        "L_total": float(total.detach().cpu()),
        "student_grad_rank": g_rank_s,
        "student_grad_embed": g_embed_s,
        "student_grad_teacher_rank": g_trank_s,
        "student_grad_weighted_embed": g_wembed_s,
        "student_grad_weighted_teacher_rank": g_wtrank_s,
        "projector_grad_embed": g_embed_p,
        "projector_grad_weighted_embed": g_wembed_p,
    })

    # Explicitly clear references; still no optimizer and no parameter update.
    del s_logits, q_s, p_s, t_logits, q_t, p_t
    del l_rank, l_embed, l_teacher_rank, weighted_embed, weighted_trank, total
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

audit_df = pd.DataFrame(rows)
display(audit_df)

# ---------------------------------------------------------
# 5. Analysis-ready summary
# ---------------------------------------------------------
metric_cols = [
    "L_rank", "L_embed", "L_teacher_rank",
    "weighted_embed_loss", "weighted_teacher_rank_loss", "L_total",
    "student_grad_rank", "student_grad_embed", "student_grad_teacher_rank",
    "student_grad_weighted_embed", "student_grad_weighted_teacher_rank",
    "projector_grad_embed", "projector_grad_weighted_embed",
]

summary = pd.DataFrame({
    "metric": metric_cols,
    "mean": [audit_df[c].mean() for c in metric_cols],
    "sd": [audit_df[c].std(ddof=1) for c in metric_cols],
    "median": [audit_df[c].median() for c in metric_cols],
    "min": [audit_df[c].min() for c in metric_cols],
    "max": [audit_df[c].max() for c in metric_cols],
})

# Ratios useful for interpreting scale balance.
eps = 1e-12
ratios = {
    "raw_loss_embed_to_teacher_rank":
        float(audit_df["L_embed"].mean() / max(audit_df["L_teacher_rank"].mean(), eps)),
    "raw_student_grad_embed_to_teacher_rank":
        float(audit_df["student_grad_embed"].mean() / max(audit_df["student_grad_teacher_rank"].mean(), eps)),
    "weighted_loss_embed_to_teacher_rank":
        float(audit_df["weighted_embed_loss"].mean() / max(audit_df["weighted_teacher_rank_loss"].mean(), eps)),
    "weighted_student_grad_embed_to_teacher_rank":
        float(audit_df["student_grad_weighted_embed"].mean() /
              max(audit_df["student_grad_weighted_teacher_rank"].mean(), eps)),
}

print("\n" + "=" * 90)
print("ANALYSIS GRADIENT / LOSS-SCALE AUDIT SUMMARY")
print("=" * 90)
print(f"checkpoint: {CKPT_PATH}")
print(f"batches: {len(audit_df)} | batch_size: {AUDIT_BATCH_SIZE}")
print(f"alpha: {alpha} | lambda_kd: {lambda_kd}")
print("\nMean component losses:")
print(audit_df[["L_rank","L_embed","L_teacher_rank",
                "weighted_embed_loss","weighted_teacher_rank_loss"]].mean())

print("\nMean student gradient norms:")
print(audit_df[["student_grad_rank","student_grad_embed",
                "student_grad_teacher_rank",
                "student_grad_weighted_embed",
                "student_grad_weighted_teacher_rank"]].mean())

print("\nScale ratios:")
print(json.dumps(ratios, indent=2))

display(summary)

# ---------------------------------------------------------
# 6. Save machine-readable evidence
# ---------------------------------------------------------
stamp = time.strftime("%Y%m%d_%H%M%S")
detail_path = AUDIT_ROOT / f"gradient_audit_batches_{stamp}.csv"
summary_path = AUDIT_ROOT / f"gradient_audit_summary_{stamp}.csv"
meta_path = AUDIT_ROOT / f"gradient_audit_metadata_{stamp}.json"

audit_df.to_csv(detail_path, index=False)
summary.to_csv(summary_path, index=False)

metadata = {
    "checkpoint": str(CKPT_PATH),
    "run_config": RUN_CFG,
    "audit_batches": len(audit_df),
    "audit_batch_size": AUDIT_BATCH_SIZE,
    "alpha": alpha,
    "lambda_kd": lambda_kd,
    "ratios": ratios,
    "no_retraining": True,
    "optimizer_created": False,
    "optimizer_step_called": False,
}
json.dump(metadata, open(meta_path, "w"), indent=2)

print("\nSaved:")
print(detail_path)
print(summary_path)
print(meta_path)

print("\nDONE. Gradient/loss-scale audit complete.")

## 11. Architecture and efficiency analysis

The efficiency audit reports exact checkpoints, transformer depth, hidden dimension, encoder parameter count, encoder state-dictionary size, and separate query/passage encoding latency. Latency uses FP16 on an NVIDIA A100, sequence length 256, 20 warm-up iterations, and 100 timed forward passes. These measurements are relative architecture comparisons, not direct measurements on resource-constrained hardware.

In [ ]:
# =========================================================
# 1. MOUNT GOOGLE DRIVE + IMPORTS + DEVICE
# =========================================================

import os
import json
import math
import time
import random
import hashlib
import gc
import shutil
from pathlib import Path
from dataclasses import dataclass, asdict

from google.colab import drive

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.stats import spearmanr

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup


# Use a dedicated empty mount point.
# This avoids the "Mountpoint must not already contain files" error.
DRIVE_MOUNT = Path("/content/peerj_stageD_drive")

if DRIVE_MOUNT.exists():
    # A stale non-mounted directory from a previous failed attempt can block mount.
    # Remove it only when it is not an active mount.
    if not os.path.ismount(str(DRIVE_MOUNT)):
        shutil.rmtree(DRIVE_MOUNT)

DRIVE_MOUNT.mkdir(parents=True, exist_ok=True)

if not os.path.ismount(str(DRIVE_MOUNT)):
    print("Mounting Google Drive for Stage D...")
    print("Choose the Google account that contains the frozen reproducible experiment files.")
    drive.mount(str(DRIVE_MOUNT))
else:
    print("Stage-D Google Drive is already mounted.")

PROJECT_ROOT_AUDIT = Path(PROJECT_DIR)

if not PROJECT_ROOT_AUDIT.exists():
    raise RuntimeError(
        f"Google Drive mount did not expose project root at: {PROJECT_ROOT_AUDIT}"
    )

print("Project root:", PROJECT_ROOT_AUDIT)
print("Drive mount check PASSED.")


# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required for this audit. "
        "In Colab select an A100/GPU runtime and rerun."
    )

DEVICE = torch.device("cuda")

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("PyTorch:", torch.__version__)
print("Device:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

In [ ]:
# =========================================================
# 4. MODEL REGISTRY + GLOBAL CONFIG
# =========================================================
MODEL_REGISTRY = {
    "T1_PubMedBERT": {"name":"microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract", "pooling":"mean", "e5":False},
    "T2_MiniLM": {"name":"sentence-transformers/all-MiniLM-L6-v2", "pooling":"mean", "e5":False},
    "T3_E5base": {"name":"intfloat/e5-base", "pooling":"mean", "e5":True},
    "S1_DistilBERT": {"name":"distilbert-base-uncased", "pooling":"mean", "e5":False},
    "S2_BioBERT": {"name":"dmis-lab/biobert-base-cased-v1.1", "pooling":"mean", "e5":False},
    "S3_MiniLM": {"name":"sentence-transformers/all-MiniLM-L6-v2", "pooling":"mean", "e5":False},
}

@dataclass
class Config:
    max_len: int = 256
    batch_size: int = 32
    eval_batch_size: int = 64
    epochs: int = 5
    lr: float = 3e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.10
    tau_rank: float = 0.05
    tau_teacher: float = 0.05
    tau_student: float = 0.05
    lambda_kd: float = 1.0
    alpha: float = 0.5
    grad_clip: float = 1.0
    geometry_top_n: int = 100
    seed: int = SEED

CFG = Config()
print(json.dumps(asdict(CFG), indent=2))
print(json.dumps(MODEL_REGISTRY, indent=2))

In [ ]:
# =========================================================
# 5. SAFE TOKENIZER + BI-ENCODER
# =========================================================
def load_tokenizer(name):
    try:
        return AutoTokenizer.from_pretrained(name, use_fast=True)
    except Exception:
        return AutoTokenizer.from_pretrained(name, use_fast=False)

def mean_pool(last_hidden, mask):
    mask = mask.unsqueeze(-1).to(last_hidden.dtype)
    return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-6)

class BiEncoder(nn.Module):
    def __init__(self, spec):
        super().__init__()
        self.spec = spec
        self.encoder = AutoModel.from_pretrained(spec["name"])
        self.out_dim = int(self.encoder.config.hidden_size)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        if self.spec["pooling"] == "mean":
            z = mean_pool(out.last_hidden_state, attention_mask)
        elif self.spec["pooling"] == "cls":
            z = out.last_hidden_state[:,0]
        else:
            raise ValueError("Unknown pooling: " + str(self.spec["pooling"]))
        return F.normalize(z, p=2, dim=-1)

def format_texts(texts, spec, kind):
    texts = [str(x) for x in texts]
    if spec.get("e5", False):
        prefix = "query: " if kind == "query" else "passage: "
        texts = [prefix + x for x in texts]
    return texts

def tokenize_texts(tok, texts, spec, kind, max_len):
    texts = format_texts(texts, spec, kind)
    return tok(texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt")

In [ ]:

# ============================================================
# ONE-CELL FINAL AUDIT — SAFE TO RERUN
# ============================================================
from pathlib import Path
import json, gc, math
import numpy as np
import pandas as pd
import torch
from transformers import AutoConfig, AutoTokenizer

ROOT = Path(PROJECT_DIR)
OUT = ROOT / "final_model_efficiency_audit_20260922"
OUT.mkdir(parents=True, exist_ok=True)

BACKBONE_CSV = OUT / "backbone_architecture_latency.csv"
RUNS_CSV = OUT / "existing_run_checkpoint_audit.csv"
ANALYSIS_CSV = OUT / "analysis_ready_model_efficiency_table.csv"
PROJ_CSV = OUT / "K1_K3_projection_audit.csv"

WARMUP = 20
REPEATS = 100
MAX_LEN = 256

QUERY_TEXT = "What are the clinical manifestations and treatment options for rheumatoid arthritis?"
PASSAGE_TEXT = (
    "Rheumatoid arthritis is a chronic systemic autoimmune disorder characterized by persistent "
    "synovial inflammation, joint pain, swelling, stiffness, and progressive structural damage. "
    "Treatment commonly includes disease-modifying antirheumatic drugs and biologic therapies."
)

def count_params(model):
    return (
        int(sum(p.numel() for p in model.parameters())),
        int(sum(p.numel() for p in model.parameters() if p.requires_grad))
    )

def infer_layers(cfg):
    for k in ("num_hidden_layers","n_layers","num_layers"):
        v = getattr(cfg, k, None)
        if v is not None:
            return int(v)
    return None

def infer_hidden(cfg):
    for k in ("hidden_size","dim","d_model"):
        v = getattr(cfg, k, None)
        if v is not None:
            return int(v)
    return None

def architecture_name(cfg):
    a = getattr(cfg, "architectures", None)
    return str(a[0]) if a else str(getattr(cfg, "model_type", "unknown"))

@torch.inference_mode()
def latency_ms(model, tok, spec, text, kind):
    enc = tokenize_texts(tok, [text], spec, kind, MAX_LEN)
    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    for _ in range(WARMUP):
        _ = model(input_ids=input_ids, attention_mask=attention_mask)
    torch.cuda.synchronize()

    vals = []
    for _ in range(REPEATS):
        st = torch.cuda.Event(enable_timing=True)
        en = torch.cuda.Event(enable_timing=True)
        st.record()
        _ = model(input_ids=input_ids, attention_mask=attention_mask)
        en.record()
        torch.cuda.synchronize()
        vals.append(float(st.elapsed_time(en)))

    return {
        "median_ms": float(np.median(vals)),
        "p95_ms": float(np.percentile(vals, 95)),
        "tokens": int(input_ids.shape[1]),
    }

# ============================================================
# A. BACKBONES
# Always construct backbone_df before section C.
# ============================================================
print("="*100)
print("A. BACKBONE ARCHITECTURE + LATENCY AUDIT")
print("="*100)
print("GPU:", torch.cuda.get_device_name(0))

backbone_rows = []

for key, spec in MODEL_REGISTRY.items():
    print(f"\nAuditing {key}: {spec['name']}")

    # --------------------------------------------------------
    # 1. TOKENIZER
    # --------------------------------------------------------
    tok = None
    tokenizer_note = "OK"

    try:
        tok = load_tokenizer(spec["name"])
    except Exception as e1:
        try:
            tok = AutoTokenizer.from_pretrained(
                spec["name"],
                use_fast=False
            )
            tokenizer_note = "slow_tokenizer_fallback"
        except Exception as e2:
            tokenizer_note = "tokenizer_unavailable"
            print(
                f"TOKENIZER WARNING [{key}]: "
                f"latency will be unavailable. {repr(e2)}"
            )

    # --------------------------------------------------------
    # 2. MODEL / ARCHITECTURE
    # --------------------------------------------------------
    model = None
    model_note = "OK"

    try:
        model = BiEncoder(spec).to(DEVICE).eval()
        cfg = model.encoder.config

        total, trainable = count_params(model)

        architecture = architecture_name(cfg)
        layers = infer_layers(cfg)
        hidden = infer_hidden(cfg)
        output_dim = int(model.out_dim)

    except Exception as e:
        # Legacy HF checkpoints such as old BioBERT may not be
        # directly loadable by current AutoConfig/AutoModel.
        model_note = "legacy_checkpoint_not_loadable_current_transformers"

        print(
            f"MODEL WARNING [{key}]: "
            f"current Transformers cannot instantiate this legacy "
            f"checkpoint. Architecture metadata will be obtained "
            f"from the registry/config where possible. {repr(e)}"
        )

        architecture = "BERT (legacy checkpoint)"
        layers = np.nan
        hidden = np.nan
        output_dim = np.nan
        total = np.nan
        trainable = np.nan

        # Try config metadata without allowing failure to stop audit
        try:
            raw_cfg_path = None

            # AutoConfig may still fail for old BioBERT.
            # Therefore this attempt is optional only.
            hf_cfg = AutoConfig.from_pretrained(
                spec["name"],
                trust_remote_code=False
            )

            architecture = architecture_name(hf_cfg)
            layers = infer_layers(hf_cfg)
            hidden = infer_hidden(hf_cfg)
            output_dim = hidden

        except Exception:
            pass

    # --------------------------------------------------------
    # 3. LATENCY
    # Only possible if BOTH tokenizer and model loaded.
    # --------------------------------------------------------
    if tok is not None and model is not None:
        try:
            q = latency_ms(
                model, tok, spec,
                QUERY_TEXT, "query"
            )

            p = latency_ms(
                model, tok, spec,
                PASSAGE_TEXT, "passage"
            )

            latency_note = "OK"

        except Exception as e:
            print(
                f"LATENCY WARNING [{key}]: "
                f"{repr(e)}"
            )

            q = {
                "tokens": np.nan,
                "median_ms": np.nan,
                "p95_ms": np.nan
            }

            p = {
                "tokens": np.nan,
                "median_ms": np.nan,
                "p95_ms": np.nan
            }

            latency_note = "latency_failed"

    else:
        q = {
            "tokens": np.nan,
            "median_ms": np.nan,
            "p95_ms": np.nan
        }

        p = {
            "tokens": np.nan,
            "median_ms": np.nan,
            "p95_ms": np.nan
        }

        latency_note = "latency_not_available"

    # --------------------------------------------------------
    # 4. SAVE ROW
    # --------------------------------------------------------
    backbone_rows.append({
        "model_key": key,
        "exact_checkpoint": spec["name"],

        "architecture": architecture,
        "transformer_layers": layers,
        "hidden_dim": hidden,
        "output_embedding_dim": output_dim,

        "pooling": spec["pooling"],

        "total_params": total,
        "trainable_params_backbone": trainable,

        "params_M": (
            total / 1e6
            if pd.notna(total)
            else np.nan
        ),

        "query_tokens": q["tokens"],
        "query_latency_median_ms": q["median_ms"],
        "query_latency_p95_ms": q["p95_ms"],

        "passage_tokens": p["tokens"],
        "passage_latency_median_ms": p["median_ms"],
        "passage_latency_p95_ms": p["p95_ms"],

        "gpu": torch.cuda.get_device_name(0),

        "warmup": WARMUP,
        "repeats": REPEATS,
        "max_len": MAX_LEN,

        "tokenizer_note": tokenizer_note,
        "model_note": model_note,
        "latency_note": latency_note,
    })

    # --------------------------------------------------------
    # 5. CLEANUP
    # --------------------------------------------------------
    if model is not None:
        del model

    if tok is not None:
        del tok

    gc.collect()
    torch.cuda.empty_cache()

backbone_df = pd.DataFrame(backbone_rows)
if backbone_df.empty:
    raise RuntimeError("Backbone audit produced no rows.")
backbone_df.to_csv(BACKBONE_CSV, index=False)
display(backbone_df)

# ============================================================
# B. EXISTING CHECKPOINTS
# ============================================================
print("\n"+"="*100)
print("B. EXISTING RUN / CHECKPOINT AUDIT")
print("="*100)

config_paths = []
for base in (ROOT/"stageD_runs", ROOT/"multiseed_runs"):
    if base.exists():
        config_paths += list(base.rglob("config.json"))

run_rows = []

for cp in sorted(config_paths):
    try:
        cfg = json.load(open(cp))
    except Exception:
        continue

    strategy = cfg.get("strategy")
    if strategy not in {"NoKD","TEACHER","K1","K2","K3"}:
        continue

    run_dir = cp.parent
    best = run_dir/"best.pt"
    last = run_dir/"last.pt"
    ckpt = best if best.exists() else (last if last.exists() else None)

    student_key = cfg.get("student_key")
    teacher_key = cfg.get("teacher_key")
    model_key = cfg.get("model_key") or (teacher_key if strategy=="TEACHER" else student_key)

    if model_key not in MODEL_REGISTRY:
        continue

    spec = MODEL_REGISTRY[model_key]
    t_spec = MODEL_REGISTRY.get(teacher_key) if teacher_key else None

    hf_cfg = AutoConfig.from_pretrained(spec["name"])
    tmp = BiEncoder(spec)
    _, backbone_trainable = count_params(tmp)
    output_dim = int(tmp.out_dim)
    del tmp

    pin = pout = None
    pparams = 0

    if ckpt is not None:
        try:
            state = torch.load(ckpt, map_location="cpu")
            ps = state.get("projector")
            if ps is not None and "weight" in ps:
                w = ps["weight"]
                pout, pin = int(w.shape[0]), int(w.shape[1])
                pparams = int(w.numel())
            del state
        except Exception as e:
            print("WARNING reading checkpoint:", ckpt, repr(e))

    # Projection is part of the embedding-alignment objective in K1/K3.
    projection_used = strategy in {"K1","K3"}
    effective_params = backbone_trainable + (pparams if projection_used else 0)

    run_rows.append({
        "strategy": strategy,
        "student_key": student_key,
        "teacher_key": teacher_key,
        "model_key": model_key,
        "exact_model_checkpoint": spec["name"],
        "exact_teacher_checkpoint": None if t_spec is None else t_spec["name"],
        "seed": cfg.get("seed"),
        "alpha": cfg.get("alpha"),
        "transformer_layers": infer_layers(hf_cfg),
        "hidden_dim": infer_hidden(hf_cfg),
        "output_embedding_dim": output_dim,
        "pooling": spec["pooling"],
        "projection_used_by_objective": projection_used,
        "projection_in_dim": pin if projection_used else None,
        "projection_out_dim": pout if projection_used else None,
        "projection_params": pparams if projection_used else 0,
        "backbone_trainable_params": backbone_trainable,
        "effective_trainable_params": effective_params,
        "checkpoint_file": None if ckpt is None else str(ckpt),
        "checkpoint_size_MB": None if ckpt is None else ckpt.stat().st_size/(1024**2),
        "run_dir": str(run_dir),
    })

runs_df = pd.DataFrame(run_rows)
if runs_df.empty:
    raise RuntimeError("No compatible Stage-D/multiseed run configs were found.")
runs_df.to_csv(RUNS_CSV, index=False)
display(runs_df)

# ============================================================
# C. ANALYSIS READY TABLE
# Re-load both CSVs deliberately so this section also works
# if the notebook state changes after A/B.
# ============================================================
print("\n"+"="*100)
print("C. ANALYSIS-READY MODEL TABLE")
print("="*100)

backbone_df = pd.read_csv(BACKBONE_CSV)
runs_df = pd.read_csv(RUNS_CSV)

rr = runs_df.copy()
rr["_seed42"] = (pd.to_numeric(rr["seed"], errors="coerce") == 42).astype(int)
rr["_stageD"] = rr["run_dir"].astype(str).str.contains("/stageD_runs/", regex=False).astype(int)

alpha_num = pd.to_numeric(rr["alpha"], errors="coerce")
rr["_main_alpha"] = ((rr["strategy"] != "K3") | ((alpha_num - 0.5).abs() < 1e-12)).astype(int)

rr = rr.sort_values(
    ["_seed42","_stageD","_main_alpha","checkpoint_size_MB"],
    ascending=[False,False,False,False],
    na_position="last"
)

mask = (
    (pd.to_numeric(rr["seed"], errors="coerce") == 42) &
    ((rr["strategy"] != "K3") | ((pd.to_numeric(rr["alpha"], errors="coerce")-0.5).abs() < 1e-12))
)
rr = rr.loc[mask].copy()

# NaN-safe duplicate key
rr["_teacher_key_safe"] = rr["teacher_key"].fillna("NONE")
rr = rr.drop_duplicates(
    subset=["strategy","model_key","_teacher_key_safe"],
    keep="first"
)

latcols = [
    "model_key",
    "query_latency_median_ms","query_latency_p95_ms",
    "passage_latency_median_ms","passage_latency_p95_ms","gpu"
]

missing = [c for c in latcols if c not in backbone_df.columns]
if missing:
    raise RuntimeError(f"Backbone CSV is missing columns: {missing}")

rr = rr.merge(backbone_df[latcols], on="model_key", how="left", validate="many_to_one")

analysis_cols = [
    "strategy","model_key","teacher_key",
    "exact_model_checkpoint","exact_teacher_checkpoint",
    "transformer_layers","hidden_dim","output_embedding_dim","pooling",
    "projection_in_dim","projection_out_dim","projection_params",
    "effective_trainable_params","checkpoint_size_MB",
    "query_latency_median_ms","query_latency_p95_ms",
    "passage_latency_median_ms","passage_latency_p95_ms",
    "gpu","checkpoint_file"
]

analysis_table = rr[analysis_cols].copy()
analysis_table.to_csv(ANALYSIS_CSV, index=False)
display(analysis_table)

projection_summary = runs_df.loc[
    runs_df["strategy"].isin(["K1","K3"]),
    ["strategy","student_key","teacher_key","seed","alpha",
     "projection_in_dim","projection_out_dim","projection_params",
     "backbone_trainable_params","effective_trainable_params","checkpoint_file"]
].copy()
projection_summary.to_csv(PROJ_CSV, index=False)

print("\n"+"="*100)
print("FINAL AUDIT COMPLETE")
print("="*100)
print("No training was run. No optimizer was created.")
print("Saved to:", OUT)
print("\nFiles:")
for p in sorted(OUT.glob("*")):
    print(" -", p.name)
print("\nAnalysis-ready model table generated.")

## 12. Statistical analysis

Repeated-seed statistics use matched seeds. MRR@10 and Recall@10 are macro-averaged across BioASQ, MedQuAD, and PubMedQA per seed before the omnibus Friedman test. Pairwise Wilcoxon signed-rank tests use Holm correction. A dedicated K3-versus-No-KD paired analysis is also provided.

In [ ]:

# ============================================================
# PeerJ FINAL PAIRED POST-HOC ANALYSIS
# K1 vs K2, K1 vs K3, K2 vs K3
# NO TRAINING — READS EXISTING 5-SEED PER-QUERY FILES ONLY
# ============================================================

from pathlib import Path
import json, math, itertools
import numpy as np
import pandas as pd
from IPython.display import display

# -----------------------------
# 0. Google Drive
# -----------------------------
from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/drive/PROJECT_ROOT")
MS_ROOT = ROOT / "multiseed_runs"
OUT = ROOT / "final_posthoc_statistics_20260922"
OUT.mkdir(parents=True, exist_ok=True)

STRATEGIES = {
    "K1": MS_ROOT / "K1",
    "K2": MS_ROOT / "K2",
    "K3": MS_ROOT / "K3_alpha010",
}
SEEDS = [42, 52, 62, 72, 82]
METRICS = ["mrr@10", "recall@10"]
PAIRS = [("K1","K2"), ("K1","K3"), ("K2","K3")]

N_PERM = 100_000
N_BOOT = 20_000
RNG_SEED = 20260922

print("Input root:", MS_ROOT)
print("Output:", OUT)

# -----------------------------
# 1. Locate and load per-query files
# -----------------------------
def find_seed_dir(strategy_root, seed):
    candidates = [
        strategy_root / f"seed_{seed}",
        strategy_root / str(seed),
    ]
    for p in candidates:
        if (p / "test_per_query.csv").exists():
            return p

    hits = list(strategy_root.rglob("test_per_query.csv")) if strategy_root.exists() else []
    for f in hits:
        # Prefer exact config seed if available
        cfg = f.parent / "config.json"
        if cfg.exists():
            try:
                c = json.load(open(cfg))
                if int(c.get("seed", -1)) == int(seed):
                    return f.parent
            except Exception:
                pass
        if str(seed) in str(f.parent):
            return f.parent
    return None

frames = []
audit_rows = []

for strategy, base in STRATEGIES.items():
    for seed in SEEDS:
        d = find_seed_dir(base, seed)
        if d is None:
            raise FileNotFoundError(
                f"Missing test_per_query.csv for {strategy}, seed={seed} under {base}"
            )

        f = d / "test_per_query.csv"
        df = pd.read_csv(f)

        required = {"source", "question_norm", *METRICS}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"{f} missing columns: {sorted(missing)}")

        x = df[["source","question_norm",*METRICS]].copy()
        x["strategy"] = strategy
        x["seed"] = seed
        frames.append(x)

        audit_rows.append({
            "strategy": strategy,
            "seed": seed,
            "rows": len(df),
            "file": str(f)
        })

audit_df = pd.DataFrame(audit_rows)
display(audit_df)
audit_df.to_csv(OUT / "input_file_audit.csv", index=False)

allq = pd.concat(frames, ignore_index=True)

# Stable paired query key
allq["query_key"] = (
    allq["source"].astype(str) + "||" + allq["question_norm"].astype(str)
)

# Check duplicates within strategy/seed/query
dup = allq.duplicated(["strategy","seed","query_key"]).sum()
if dup:
    raise ValueError(f"Found {dup} duplicate strategy/seed/query rows.")

print("\nLoaded rows:", len(allq))
print("Unique queries:", allq["query_key"].nunique())

# -----------------------------
# 2. Verify exact pairing across all strategies/seeds
# -----------------------------
keysets = {}
for strategy in STRATEGIES:
    for seed in SEEDS:
        s = set(allq.loc[
            (allq.strategy == strategy) & (allq.seed == seed),
            "query_key"
        ])
        keysets[(strategy,seed)] = s

reference = keysets[("K1",42)]
mismatches = []
for k, s in keysets.items():
    if s != reference:
        mismatches.append({
            "strategy": k[0], "seed": k[1],
            "missing_vs_reference": len(reference - s),
            "extra_vs_reference": len(s - reference)
        })

if mismatches:
    display(pd.DataFrame(mismatches))
    raise ValueError("Per-query sets are not identical; paired analysis aborted.")

print("PAIRING CHECK: PASSED")
print("Paired queries per run:", len(reference))

# -----------------------------
# 3. Average each query over the 5 seeds
#    This is the primary query-level paired analysis.
# -----------------------------
avgq = (
    allq.groupby(["strategy","query_key","source"], as_index=False)[METRICS]
        .mean()
)

wide = {}
for metric in METRICS:
    w = avgq.pivot(index=["query_key","source"], columns="strategy", values=metric)
    w = w[["K1","K2","K3"]].dropna()
    wide[metric] = w
    w.reset_index().to_csv(OUT / f"query_scores_5seed_mean_{metric.replace('@','at')}.csv", index=False)

# -----------------------------
# 4. Statistics helpers
# -----------------------------
def paired_randomization_p(diff, n_perm=100_000, seed=0):
    """Two-sided paired sign randomization on query-level differences."""
    diff = np.asarray(diff, dtype=np.float64)
    obs = abs(diff.mean())
    rng = np.random.default_rng(seed)

    # Chunked to keep memory bounded.
    extreme = 0
    done = 0
    chunk = 2000
    while done < n_perm:
        m = min(chunk, n_perm - done)
        signs = rng.choice(np.array([-1.0, 1.0]), size=(m, len(diff)))
        stats = np.abs((signs * diff).mean(axis=1))
        extreme += int(np.sum(stats >= obs - 1e-15))
        done += m

    return (extreme + 1) / (n_perm + 1)

def bootstrap_mean_ci(diff, n_boot=20_000, seed=0):
    diff = np.asarray(diff, dtype=np.float64)
    rng = np.random.default_rng(seed)
    n = len(diff)

    vals = np.empty(n_boot, dtype=np.float64)
    chunk = 1000
    done = 0
    while done < n_boot:
        m = min(chunk, n_boot - done)
        idx = rng.integers(0, n, size=(m, n))
        vals[done:done+m] = diff[idx].mean(axis=1)
        done += m

    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(lo), float(hi)

def cohen_dz(diff):
    diff = np.asarray(diff, dtype=np.float64)
    sd = diff.std(ddof=1)
    return float(diff.mean()/sd) if sd > 0 else np.nan

def holm_adjust(pvals):
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    adjusted_sorted = np.empty(m, dtype=float)

    running = 0.0
    for rank, idx in enumerate(order):
        val = (m-rank) * pvals[idx]
        running = max(running, val)
        adjusted_sorted[rank] = min(running, 1.0)

    out = np.empty(m, dtype=float)
    for rank, idx in enumerate(order):
        out[idx] = adjusted_sorted[rank]
    return out

# -----------------------------
# 5. PRIMARY: query-level paired post-hoc
# -----------------------------
rows = []

for metric_i, metric in enumerate(METRICS):
    w = wide[metric]

    metric_rows = []
    for pair_i, (a,b) in enumerate(PAIRS):
        # Delta is second minus first, so positive => B better than A.
        diff = (w[b] - w[a]).to_numpy(dtype=float)

        ci_lo, ci_hi = bootstrap_mean_ci(
            diff, N_BOOT, RNG_SEED + 1000*metric_i + pair_i
        )
        p = paired_randomization_p(
            diff, N_PERM, RNG_SEED + 2000*metric_i + pair_i
        )

        wins = int(np.sum(diff > 1e-12))
        losses = int(np.sum(diff < -1e-12))
        ties = int(len(diff) - wins - losses)

        metric_rows.append({
            "metric": metric,
            "comparison": f"{a} vs {b}",
            "A": a,
            "B": b,
            "n_paired_queries": len(diff),
            "A_mean": float(w[a].mean()),
            "B_mean": float(w[b].mean()),
            "delta_B_minus_A": float(diff.mean()),
            "bootstrap_95CI_low": ci_lo,
            "bootstrap_95CI_high": ci_hi,
            "cohen_dz": cohen_dz(diff),
            "wins_B": wins,
            "ties": ties,
            "wins_A": losses,
            "p_randomization_raw": p,
        })

    adj = holm_adjust([r["p_randomization_raw"] for r in metric_rows])
    for r, p_adj in zip(metric_rows, adj):
        r["p_holm"] = float(p_adj)
        r["significant_holm_0.05"] = bool(p_adj < 0.05)
        rows.append(r)

primary = pd.DataFrame(rows)
primary.to_csv(OUT / "primary_query_level_posthoc.csv", index=False)

print("\n" + "="*110)
print("PRIMARY QUERY-LEVEL PAIRED POST-HOC ANALYSIS")
print("5-seed mean score per query; paired randomization; Holm correction within each metric")
print("="*110)
display(primary)

# -----------------------------
# 6. Per-source complementary analysis
# -----------------------------
source_rows = []

for metric_i, metric in enumerate(METRICS):
    w = wide[metric].reset_index()

    for source_i, source in enumerate(sorted(w["source"].unique())):
        ws = w[w["source"] == source].set_index("query_key")

        temp = []
        for pair_i, (a,b) in enumerate(PAIRS):
            diff = (ws[b] - ws[a]).to_numpy(float)
            ci_lo, ci_hi = bootstrap_mean_ci(
                diff, N_BOOT,
                RNG_SEED + 10_000 + 1000*metric_i + 100*source_i + pair_i
            )
            p = paired_randomization_p(
                diff, N_PERM,
                RNG_SEED + 20_000 + 1000*metric_i + 100*source_i + pair_i
            )
            temp.append({
                "source": source,
                "metric": metric,
                "comparison": f"{a} vs {b}",
                "n_queries": len(diff),
                "delta_B_minus_A": float(diff.mean()),
                "bootstrap_95CI_low": ci_lo,
                "bootstrap_95CI_high": ci_hi,
                "cohen_dz": cohen_dz(diff),
                "p_randomization_raw": p,
            })

        # Holm within source + metric across the 3 strategy comparisons
        adj = holm_adjust([r["p_randomization_raw"] for r in temp])
        for r, p_adj in zip(temp, adj):
            r["p_holm"] = float(p_adj)
            r["significant_holm_0.05"] = bool(p_adj < 0.05)
            source_rows.append(r)

per_source = pd.DataFrame(source_rows)
per_source.to_csv(OUT / "complementary_per_source_posthoc.csv", index=False)

print("\n" + "="*110)
print("COMPLEMENTARY PER-SOURCE ANALYSIS")
print("="*110)
display(per_source)

# -----------------------------
# 7. SECONDARY: seed-level macro descriptive paired analysis
#    (not used as the sole significance test because n=5 is very small)
# -----------------------------
seed_macro = (
    allq.groupby(["strategy","seed"], as_index=False)[METRICS]
        .mean()
)

seed_rows = []
for metric in METRICS:
    sw = seed_macro.pivot(index="seed", columns="strategy", values=metric)
    for a,b in PAIRS:
        d = (sw[b] - sw[a]).to_numpy(float)
        seed_rows.append({
            "metric": metric,
            "comparison": f"{a} vs {b}",
            "n_seeds": len(d),
            "mean_delta_B_minus_A": float(d.mean()),
            "sd_delta": float(d.std(ddof=1)),
            "seeds_B_better": int(np.sum(d > 0)),
            "seeds_A_better": int(np.sum(d < 0)),
            "ties": int(np.sum(d == 0)),
        })

seed_summary = pd.DataFrame(seed_rows)
seed_summary.to_csv(OUT / "secondary_seed_level_descriptive.csv", index=False)

print("\n" + "="*110)
print("SECONDARY 5-SEED DESCRIPTIVE SUMMARY")
print("="*110)
display(seed_summary)

# -----------------------------
# 8. Analysis-ready compact table
# -----------------------------
analysis = primary[[
    "metric","comparison","n_paired_queries",
    "A_mean","B_mean","delta_B_minus_A",
    "bootstrap_95CI_low","bootstrap_95CI_high",
    "cohen_dz","p_randomization_raw","p_holm",
    "significant_holm_0.05"
]].copy()

analysis.to_csv(OUT / "analysis_ready_posthoc_table.csv", index=False)

print("\n" + "="*110)
print("ANALYSIS-READY POST-HOC TABLE")
print("="*110)
display(analysis)

# -----------------------------
# 9. Plain-text interpretation
# -----------------------------
lines = []
lines.append("FINAL POST-HOC STATISTICAL SUMMARY")
lines.append("="*80)
lines.append(
    "Primary inference: query-level paired analysis after averaging each query over the five seeds; "
    "two-sided paired randomization; Holm correction across K1/K2/K3 pairwise comparisons within each metric."
)

for metric in METRICS:
    lines.append("")
    lines.append(metric.upper())
    for _, r in analysis[analysis.metric == metric].iterrows():
        sig = "SIGNIFICANT" if r["significant_holm_0.05"] else "not significant"
        lines.append(
            f"{r['comparison']}: delta={r['delta_B_minus_A']:+.6f}, "
            f"95% CI [{r['bootstrap_95CI_low']:+.6f}, {r['bootstrap_95CI_high']:+.6f}], "
            f"dz={r['cohen_dz']:+.4f}, raw p={r['p_randomization_raw']:.6g}, "
            f"Holm p={r['p_holm']:.6g} -> {sig}."
        )

summary_text = "\n".join(lines)
print("\n" + summary_text)

with open(OUT / "FINAL_POSTHOC_SUMMARY.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)

print("\nSaved files:")
for p in sorted(OUT.iterdir()):
    print(" -", p.name)

print("\nANALYSIS COMPLETE. NO TRAINING WAS RUN.")

### 12.1 K3 versus No-KD paired comparison

In [ ]:
# =========================================================
# 1. MOUNT DRIVE SAFELY
# =========================================================
from pathlib import Path
import os

try:
    from google.colab import drive
    if not Path(PROJECT_DIR).exists():
        drive.mount("/content/drive")
    else:
        print("Google Drive already mounted.")
except Exception as e:
    print("Drive mount note:", e)

BASE = Path("/content/drive/PROJECT_ROOT")
assert BASE.exists(), f"Revision directory not found: {BASE}"
print("BASE:", BASE)

In [ ]:
# =========================================================
# 2. IMPORTS + SETTINGS
# =========================================================
import json, re, math, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEEDS = [42, 52, 62, 72, 82]
METRICS = ["mrr@1", "mrr@5", "mrr@10", "recall@1", "recall@5", "recall@10"]
PRIMARY_METRICS = ["mrr@10", "recall@10"]

N_PERMUTATIONS = 20000
N_BOOTSTRAP = 20000
RANDOM_SEED = 20260922

OUTDIR = BASE / "significance" / "K3_alpha010_vs_NoKD"
OUTDIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTDIR)
print("Seeds:", SEEDS)

In [ ]:
# =========================================================
# 3. DISCOVER ALL PER-QUERY FILES
# =========================================================
def all_per_query_files(base):
    return sorted(base.rglob("test_per_query.csv"))

all_files = all_per_query_files(BASE)
print(f"Found {len(all_files)} test_per_query.csv files.\n")
for p in all_files:
    print(p)

In [ ]:
# =========================================================
# 4. AUTOMATICALLY CLASSIFY NoKD AND K3 α=0.10 FILES
#    Uses path + nearby config/metadata where possible.
# =========================================================
def load_json_safe(path):
    try:
        return json.loads(path.read_text())
    except Exception:
        return {}

def nearby_jsons(csv_path):
    candidates = [
        csv_path.parent / "config.json",
        csv_path.parent / "COMPLETE.json",
        csv_path.parent.parent / "config.json",
        csv_path.parent.parent / "COMPLETE.json",
    ]
    return {p.name: load_json_safe(p) for p in candidates if p.exists()}

def infer_seed(path, metadata):
    # metadata first
    for d in metadata.values():
        if isinstance(d, dict) and "seed" in d:
            try: return int(d["seed"])
            except: pass
    # path patterns
    s = str(path)
    pats = [r"seed[_-]?(\d+)", r"_s(\d+)(?:_|/|$)"]
    for pat in pats:
        m = re.search(pat, s, flags=re.I)
        if m:
            return int(m.group(1))
    return None

def flatten_meta(metadata):
    vals = []
    def walk(x):
        if isinstance(x, dict):
            for k,v in x.items():
                vals.append(str(k)); walk(v)
        elif isinstance(x, (list,tuple)):
            for v in x: walk(v)
        else:
            vals.append(str(x))
    walk(metadata)
    return " ".join(vals).lower()

records = []
for p in all_files:
    meta = nearby_jsons(p)
    txt = (str(p) + " " + flatten_meta(meta)).lower()
    seed = infer_seed(p, meta)

    is_nokd = ("nokd" in txt or "rank_loss_only" in txt) and seed in SEEDS

    # K3 alpha .10 detection: path/config conventions
    is_k3 = ("k3" in txt) and seed in SEEDS
    alpha010 = any(tok in txt for tok in [
        "alpha010", "alpha0.10", "alpha_0.10", "a0.10",
        '"alpha" 0.1', "'alpha' 0.1", "alpha 0.1"
    ])
    # inspect config explicitly
    for d in meta.values():
        if isinstance(d, dict):
            a = d.get("alpha", None)
            strategy = str(d.get("strategy","")).lower()
            if a is not None:
                try:
                    if abs(float(a)-0.1) < 1e-12:
                        alpha010 = True
                except: pass
            if strategy == "k3":
                is_k3 = True

    records.append({
        "path": str(p),
        "seed": seed,
        "is_nokd": is_nokd,
        "is_k3_alpha010": bool(is_k3 and alpha010),
    })

discovery = pd.DataFrame(records)
display(discovery)
discovery.to_csv(OUTDIR / "discovered_per_query_files.csv", index=False)

nokd_candidates = discovery[discovery.is_nokd]
k3_candidates = discovery[discovery.is_k3_alpha010]

print("\nNoKD candidates:", len(nokd_candidates))
display(nokd_candidates)
print("\nK3 alpha=0.10 candidates:", len(k3_candidates))
display(k3_candidates)

In [ ]:
# =========================================================
# 5. RESOLVE EXACTLY ONE FILE PER SEED
#    If auto-discovery is ambiguous, edit MANUAL_* below.
# =========================================================
MANUAL_NOKD = {
    # 42: "/content/drive/.../test_per_query.csv",
}
MANUAL_K3 = {
    # 42: "/content/drive/.../test_per_query.csv",
}

def resolve_by_seed(candidates, manual, label):
    out = {}
    for seed in SEEDS:
        if seed in manual:
            p = Path(manual[seed])
            assert p.exists(), f"{label} seed {seed}: manual path missing: {p}"
            out[seed] = p
            continue
        sub = candidates[candidates.seed == seed]
        if len(sub) == 1:
            out[seed] = Path(sub.iloc[0].path)
        elif len(sub) == 0:
            raise FileNotFoundError(
                f"{label} seed {seed}: no test_per_query.csv found. "
                f"Inspect the discovery table above and add its path to MANUAL_{label.upper().replace(' ','_')}."
            )
        else:
            raise RuntimeError(
                f"{label} seed {seed}: {len(sub)} candidates found; ambiguous.\n"
                + "\n".join(sub.path.tolist())
            )
    return out

NOKD_FILES = resolve_by_seed(nokd_candidates, MANUAL_NOKD, "NoKD")
K3_FILES   = resolve_by_seed(k3_candidates, MANUAL_K3, "K3")

print("="*90)
print("RESOLVED FILES")
print("="*90)
for s in SEEDS:
    print(f"seed {s}")
    print("  NoKD:", NOKD_FILES[s])
    print("  K3:  ", K3_FILES[s])

In [ ]:
# =========================================================
# 6. LOAD + SCHEMA AUDIT
# =========================================================
def read_query_file(path):
    df = pd.read_csv(path)
    return df

for seed in SEEDS:
    a = read_query_file(NOKD_FILES[seed])
    b = read_query_file(K3_FILES[seed])
    print("\n", "="*90)
    print("SEED", seed)
    print("NoKD:", a.shape, list(a.columns))
    print("K3:  ", b.shape, list(b.columns))

print("\nSchema audit displayed. Next cell infers the pairing key.")

In [ ]:
# =========================================================
# 7. INFER STABLE QUERY PAIRING KEY
# =========================================================
PREFERRED_KEYS = [
    ["source", "question"],
    ["source", "query"],
    ["source", "question_id"],
    ["source", "query_id"],
    ["question"],
    ["query"],
    ["question_id"],
    ["query_id"],
]

def infer_key(a, b):
    common = set(a.columns) & set(b.columns)
    for ks in PREFERRED_KEYS:
        if all(k in common for k in ks):
            # require uniqueness in each file
            if not a.duplicated(ks).any() and not b.duplicated(ks).any():
                return ks
    raise ValueError(
        "Could not infer a unique query key. Common columns: "
        + ", ".join(sorted(common))
    )

KEYS = {}
for seed in SEEDS:
    a = read_query_file(NOKD_FILES[seed])
    b = read_query_file(K3_FILES[seed])
    KEYS[seed] = infer_key(a,b)
    print(f"seed {seed}: pairing key = {KEYS[seed]}")

In [ ]:
# =========================================================
# 8. EXACT ALIGNMENT AUDIT + MERGED QUERY TABLES
# =========================================================
MERGED = {}
audit_rows = []

for seed in SEEDS:
    a = read_query_file(NOKD_FILES[seed])
    b = read_query_file(K3_FILES[seed])
    keys = KEYS[seed]

    common_metrics = [m for m in METRICS if m in a.columns and m in b.columns]
    assert "mrr@10" in common_metrics, f"seed {seed}: mrr@10 missing"
    assert "recall@10" in common_metrics, f"seed {seed}: recall@10 missing"

    left_keys = set(map(tuple, a[keys].astype(str).to_numpy()))
    right_keys = set(map(tuple, b[keys].astype(str).to_numpy()))
    exact = left_keys == right_keys

    merged = a[keys + common_metrics].merge(
        b[keys + common_metrics],
        on=keys, how="outer", suffixes=("_nokd", "_k3"), indicator=True
    )
    both = (merged["_merge"] == "both").all()

    audit_rows.append({
        "seed": seed,
        "nokd_rows": len(a),
        "k3_rows": len(b),
        "pairing_key": "+".join(keys),
        "exact_key_set_match": exact,
        "all_rows_matched": both,
        "merged_rows": len(merged),
    })

    if not exact or not both:
        bad = merged[merged["_merge"] != "both"]
        bad.to_csv(OUTDIR / f"alignment_mismatch_seed{seed}.csv", index=False)
        raise AssertionError(
            f"Seed {seed}: K3 and NoKD query sets are not identical. "
            f"Mismatch file saved."
        )

    merged = merged.drop(columns="_merge")
    if "source" not in merged.columns:
        raise ValueError("A 'source' column is required for source-level analysis.")

    for m in common_metrics:
        merged[f"delta_{m}"] = merged[f"{m}_k3"] - merged[f"{m}_nokd"]

    MERGED[seed] = merged
    merged.to_csv(OUTDIR / f"paired_queries_seed{seed}.csv", index=False)

audit_df = pd.DataFrame(audit_rows)
display(audit_df)
audit_df.to_csv(OUTDIR / "query_alignment_audit.csv", index=False)

assert audit_df["exact_key_set_match"].all()
assert audit_df["all_rows_matched"].all()
print("\nEXACT QUERY ALIGNMENT: PASSED")

In [ ]:
# =========================================================
# 9. STATISTICAL FUNCTIONS
# =========================================================
rng = np.random.default_rng(RANDOM_SEED)

def paired_bootstrap_delta(x, y, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED):
    """Delta = mean(K3 - NoKD); paired bootstrap over queries."""
    x = np.asarray(x, dtype=float)  # NoKD
    y = np.asarray(y, dtype=float)  # K3
    d = y - x
    n = len(d)
    local = np.random.default_rng(seed)
    # chunk to control memory
    vals = []
    chunk = 1000
    for start in range(0, n_boot, chunk):
        k = min(chunk, n_boot-start)
        idx = local.integers(0, n, size=(k,n))
        vals.append(d[idx].mean(axis=1))
    boot = np.concatenate(vals)
    lo, hi = np.quantile(boot, [0.025, 0.975])
    return d.mean(), lo, hi

def paired_signflip_test(x, y, n_perm=N_PERMUTATIONS, seed=RANDOM_SEED):
    """Two-sided paired randomization/sign-flip test on mean difference."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    d = y - x
    obs = abs(d.mean())
    local = np.random.default_rng(seed)
    exceed = 0
    done = 0
    chunk = 1000
    while done < n_perm:
        k = min(chunk, n_perm-done)
        signs = local.choice(np.array([-1.0,1.0]), size=(k,len(d)))
        perm_means = np.abs((signs*d).mean(axis=1))
        exceed += int((perm_means >= obs - 1e-15).sum())
        done += k
    # +1 correction
    return (exceed + 1) / (n_perm + 1)

def stars(p):
    if p < 0.001: return "***"
    if p < 0.01: return "**"
    if p < 0.05: return "*"
    return "ns"

In [ ]:
# =========================================================
# 10. PER-SEED, PER-SOURCE PAIRED TESTS
# =========================================================
rows = []

for seed in SEEDS:
    df = MERGED[seed]
    groups = [(s, g) for s,g in df.groupby("source")]
    groups.append(("ALL_QUERIES", df))

    for source, g in groups:
        for metric in PRIMARY_METRICS:
            x = g[f"{metric}_nokd"].to_numpy(float)
            y = g[f"{metric}_k3"].to_numpy(float)

            delta, lo, hi = paired_bootstrap_delta(
                x, y, seed=RANDOM_SEED + seed + len(rows)
            )
            p = paired_signflip_test(
                x, y, seed=RANDOM_SEED + 10000 + seed + len(rows)
            )

            rows.append({
                "seed": seed,
                "source": source,
                "metric": metric,
                "n_queries": len(g),
                "nokd_mean": x.mean(),
                "k3_mean": y.mean(),
                "delta_k3_minus_nokd": delta,
                "bootstrap_ci95_low": lo,
                "bootstrap_ci95_high": hi,
                "paired_randomization_p": p,
                "significance": stars(p),
            })

per_seed_stats = pd.DataFrame(rows)
display(per_seed_stats)
per_seed_stats.to_csv(OUTDIR / "per_seed_paired_significance.csv", index=False)

In [ ]:
# =========================================================
# 11. SOURCE-MACRO METRICS PER SEED
#     Reproduces the paper's macro-over-datasets perspective.
# =========================================================
macro_rows = []

for seed in SEEDS:
    df = MERGED[seed]
    for metric in PRIMARY_METRICS:
        source_means = []
        for source, g in df.groupby("source"):
            source_means.append({
                "source": source,
                "nokd": g[f"{metric}_nokd"].mean(),
                "k3": g[f"{metric}_k3"].mean()
            })
        sm = pd.DataFrame(source_means)
        macro_rows.append({
            "seed": seed,
            "metric": metric,
            "nokd_macro": sm["nokd"].mean(),
            "k3_macro": sm["k3"].mean(),
            "delta_k3_minus_nokd": (sm["k3"] - sm["nokd"]).mean()
        })

macro_seed = pd.DataFrame(macro_rows)
display(macro_seed)
macro_seed.to_csv(OUTDIR / "macro_per_seed_differences.csv", index=False)

In [ ]:
# =========================================================
# 12. 5-SEED PAIRED INFERENCE ON MACRO SCORES
#     Exact sign-flip test is possible with n=5 (2^5=32).
# =========================================================
from itertools import product

def exact_signflip_p(d):
    d = np.asarray(d, float)
    obs = abs(d.mean())
    vals = []
    for signs in product([-1.0,1.0], repeat=len(d)):
        vals.append(abs((d*np.array(signs)).mean()))
    vals = np.asarray(vals)
    return float((vals >= obs - 1e-15).mean())

def seed_bootstrap_ci(d, n_boot=50000, seed=RANDOM_SEED):
    d = np.asarray(d,float)
    local = np.random.default_rng(seed)
    idx = local.integers(0, len(d), size=(n_boot,len(d)))
    b = d[idx].mean(axis=1)
    return np.quantile(b,[0.025,0.975])

seed_summary_rows = []
for metric in PRIMARY_METRICS:
    s = macro_seed[macro_seed.metric == metric].copy()
    d = s["delta_k3_minus_nokd"].to_numpy(float)
    lo, hi = seed_bootstrap_ci(d, seed=RANDOM_SEED + len(seed_summary_rows))
    p = exact_signflip_p(d)
    seed_summary_rows.append({
        "metric": metric,
        "n_seeds": len(d),
        "nokd_macro_mean": s["nokd_macro"].mean(),
        "nokd_macro_sd": s["nokd_macro"].std(ddof=1),
        "k3_macro_mean": s["k3_macro"].mean(),
        "k3_macro_sd": s["k3_macro"].std(ddof=1),
        "mean_delta_k3_minus_nokd": d.mean(),
        "delta_seed_bootstrap_ci95_low": lo,
        "delta_seed_bootstrap_ci95_high": hi,
        "exact_seed_signflip_p": p,
        "significance": stars(p),
        "k3_better_seeds": int((d>0).sum()),
        "nokd_better_seeds": int((d<0).sum()),
        "ties": int((d==0).sum()),
    })

seed_summary = pd.DataFrame(seed_summary_rows)
display(seed_summary)
seed_summary.to_csv(OUTDIR / "five_seed_macro_paired_summary.csv", index=False)

In [ ]:
# =========================================================
# 13. POOLED QUERY-LEVEL ANALYSIS ACROSS SEEDS
#     Treats each seed-query result as a paired observation.
#     Reported as a complementary analysis, not a replacement
#     for the 5-seed macro comparison.
# =========================================================
pooled_rows = []

for metric in PRIMARY_METRICS:
    for source in sorted(MERGED[SEEDS[0]]["source"].unique()):
        xs, ys = [], []
        for seed in SEEDS:
            g = MERGED[seed]
            g = g[g.source == source]
            xs.append(g[f"{metric}_nokd"].to_numpy(float))
            ys.append(g[f"{metric}_k3"].to_numpy(float))
        x = np.concatenate(xs); y = np.concatenate(ys)
        delta, lo, hi = paired_bootstrap_delta(
            x,y,seed=RANDOM_SEED + len(pooled_rows)
        )
        p = paired_signflip_test(
            x,y,seed=RANDOM_SEED + 20000 + len(pooled_rows)
        )
        pooled_rows.append({
            "source": source,
            "metric": metric,
            "paired_seed_query_observations": len(x),
            "nokd_mean": x.mean(),
            "k3_mean": y.mean(),
            "delta_k3_minus_nokd": delta,
            "bootstrap_ci95_low": lo,
            "bootstrap_ci95_high": hi,
            "paired_randomization_p": p,
            "significance": stars(p),
        })

pooled = pd.DataFrame(pooled_rows)
display(pooled)
pooled.to_csv(OUTDIR / "pooled_seed_query_significance.csv", index=False)

In [ ]:
# =========================================================
# 14. ANALYSIS-READY TABLE
# =========================================================
review_rows = []

for metric in PRIMARY_METRICS:
    s = seed_summary[seed_summary.metric == metric].iloc[0]
    review_rows.append({
        "comparison": "K3 alpha=0.10 vs NoKD",
        "metric": metric,
        "NoKD mean±SD": f'{s["nokd_macro_mean"]:.4f} ± {s["nokd_macro_sd"]:.4f}',
        "K3 mean±SD": f'{s["k3_macro_mean"]:.4f} ± {s["k3_macro_sd"]:.4f}',
        "Delta (K3-NoKD)": f'{s["mean_delta_k3_minus_nokd"]:+.4f}',
        "95% CI of Delta": f'[{s["delta_seed_bootstrap_ci95_low"]:+.4f}, {s["delta_seed_bootstrap_ci95_high"]:+.4f}]',
        "Exact paired p": f'{s["exact_seed_signflip_p"]:.4f}',
        "Significance": s["significance"],
        "K3 better seeds": int(s["k3_better_seeds"]),
        "NoKD better seeds": int(s["nokd_better_seeds"]),
    })

analysis = pd.DataFrame(review_rows)
display(analysis)
analysis.to_csv(OUTDIR / "analysis_ready_K3_vs_NoKD.csv", index=False)

print("\nSaved:", OUTDIR / "analysis_ready_K3_vs_NoKD.csv")

In [ ]:
# =========================================================
# 15. AUTOMATIC INTERPRETATION SUMMARY
# =========================================================
print("="*90)
print("K3 alpha=0.10 vs NoKD — FINAL STATISTICAL SUMMARY")
print("="*90)

for _, r in seed_summary.iterrows():
    metric = r["metric"]
    print(f"\n{metric}")
    print(f'  NoKD macro: {r["nokd_macro_mean"]:.6f} ± {r["nokd_macro_sd"]:.6f}')
    print(f'  K3 macro:   {r["k3_macro_mean"]:.6f} ± {r["k3_macro_sd"]:.6f}')
    print(f'  Delta:      {r["mean_delta_k3_minus_nokd"]:+.6f}')
    print(f'  95% CI:     [{r["delta_seed_bootstrap_ci95_low"]:+.6f}, {r["delta_seed_bootstrap_ci95_high"]:+.6f}]')
    print(f'  Exact paired sign-flip p: {r["exact_seed_signflip_p"]:.6f}')
    print(f'  Seeds K3>NoKD: {int(r["k3_better_seeds"])}/5')
    if r["exact_seed_signflip_p"] < 0.05:
        print("  INTERPRETATION: statistically significant at p < 0.05.")
    else:
        print("  INTERPRETATION: NOT statistically significant at p < 0.05.")

print("\nIMPORTANT:")
print("- Primary robustness inference should emphasize the 5-seed paired macro analysis.")
print("- Query-level pooled/source analyses are complementary.")
print("- Do not claim statistical significance unless the corresponding p-value supports it.")

In [ ]:
# =========================================================
# 16. MANIFEST + COMPLETION MARKER
# =========================================================
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

outputs = sorted(OUTDIR.glob("*.csv"))
manifest = {
    "analysis": "K3_alpha010_vs_NoKD_paired_significance",
    "seeds": SEEDS,
    "primary_metrics": PRIMARY_METRICS,
    "n_permutations_query_level": N_PERMUTATIONS,
    "n_bootstrap_query_level": N_BOOTSTRAP,
    "random_seed": RANDOM_SEED,
    "NoKD_files": {str(k): str(v) for k,v in NOKD_FILES.items()},
    "K3_files": {str(k): str(v) for k,v in K3_FILES.items()},
    "outputs": {
        p.name: {"sha256": sha256_file(p), "bytes": p.stat().st_size}
        for p in outputs
    }
}

(OUTDIR / "analysis_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
(OUTDIR / "COMPLETE.txt").write_text(
    "K3 alpha=0.10 vs NoKD paired significance analysis completed successfully.\n",
    encoding="utf-8"
)

print("="*90)
print("ANALYSIS COMPLETE")
print("="*90)
print("Output directory:", OUTDIR)
print("\nGenerated files:")
for p in sorted(OUTDIR.iterdir()):
    print(" -", p.name)

## Submission preflight checks

This final preflight verifies the key protocol constants and guards against accidental reintroduction of user-specific paths. It does not rerun training.


In [ ]:
assert CFG.max_len == 256
assert CFG.batch_size == 32
assert CFG.epochs == 5
assert abs(CFG.lr - 3e-5) < 1e-12
assert abs(CFG.warmup_ratio - 0.10) < 1e-12
assert abs(CFG.tau_rank - 0.05) < 1e-12
assert abs(CFG.tau_teacher - 0.05) < 1e-12
assert abs(CFG.tau_student - 0.05) < 1e-12
assert abs(CFG.lambda_kd - 1.0) < 1e-12
assert ALPHAS == [0.00, 0.10, 0.30, 0.50, 0.70, 0.90, 1.00]
assert SEEDS == [42, 52, 62, 72, 82]
assert MAIN_K3_ALPHA == 0.50
assert not any(token in str(PROJECT_DIR) for token in ["MyDrive/PhD", "Amine", "Gonca", "Toprak"])
print("Submission preflight checks PASSED.")


## 13. Reproducibility outputs

For submission, retain the fixed split CSV files, structured training CSV, machine-readable run configurations, per-seed result files, and summary tables generated by this notebook. Checkpoints are generated by the training sections and are not required to be embedded in the notebook itself.

In [ ]:
manifest = {
    "project_root": str(PROJECT_DIR),
    "seeds": [42, 52, 62, 72, 82],
    "max_length": CFG.max_len,
    "batch_size": CFG.batch_size,
    "epochs": CFG.epochs,
    "learning_rate": CFG.lr,
    "warmup_ratio": CFG.warmup_ratio,
    "tau_rank": CFG.tau_rank,
    "tau_teacher": CFG.tau_teacher,
    "tau_student": CFG.tau_student,
    "lambda_kd": CFG.lambda_kd,
    "multiseed_k3_alpha": 0.10,
    "geometry_top_n": 100,
    "models": MODEL_REGISTRY,
}
manifest_path = Path(CONFIG_DIR) / "submission_reproducibility_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print("Saved:", manifest_path)